In [ ]:
# -------------------------
# 1) Basic Library Import
# -------------------------
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input, Sequential, optimizers
import time
from tqdm import tqdm

In [ ]:
# -------------------------
# 2) Mortality Rates Table
# -------------------------
def parse_hmd_file(filepath, start_year=None):
    """
    Parses a standard 1x1 data file from the Human Mortality Database (HMD) for Switzerland.
    
    Args:
        filepath (str): The path to the HMD .txt file.
        start_year (int, optional): The first year to include in the data. Defaults to None.
        
    Returns:
        (pd.DataFrame, np.ndarray, np.ndarray): A tuple containing:
            - mx_matrix_df: A DataFrame of central death rates (mx) with ages as rows and years as columns.
            - ages: A numpy array of the age labels.
            - years: A numpy array of the year labels.
    """
    try:
        with open(filepath, 'r') as f:
            for i, line in enumerate(f):
                if line.strip().startswith('Year'):
                    header_line = i
                    break
        data = pd.read_csv(
            filepath,
            skiprows=header_line,
            delim_whitespace=True,
            na_values=['.']
        )
        mx_matrix_df = data.pivot(index='Age', columns='Year', values='Total')
        mx_matrix_df.index = pd.to_numeric(mx_matrix_df.index.str.replace('+', ''), errors='coerce')
        mx_matrix_df = mx_matrix_df.sort_index()
        mx_matrix_df = mx_matrix_df.dropna()
        if start_year is not None:
            mx_matrix_df.columns = pd.to_numeric(mx_matrix_df.columns)
            mx_matrix_df = mx_matrix_df.loc[:, mx_matrix_df.columns >= start_year]
        ages = mx_matrix_df.index.values
        years = mx_matrix_df.columns.values

        print(f"Successfully loaded data for ages {int(ages.min())}-{int(ages.max())} from {int(years.min())}-{int(years.max())}.")
        
        return mx_matrix_df, ages, years
    
    except FileNotFoundError:
        print(f"Error: The file '{filepath}' was not found.")
        return None, None, None
    except Exception as e:
        print(f"An error occurred while parsing the file: {e}")
        return None, None, None

hmd_filepath = 'Switzerland_mortality_rates.txt'
mx_df, ages_hmd, years_hmd = parse_hmd_file(hmd_filepath, start_year=1950)

In [ ]:
# -------------------------
# 3) Lee-Carter Model for Stochastic Mortality
# -------------------------
from scipy.optimize import minimize

class LeeCarter:
    def __init__(self):
        self.a_x = None
        self.b_x = None
        self.k_t = None
        self.ages = None
        self.years = None
        self.drift = None
        self.volatility = None
        
    def fit(self, mx_df, ages, years):
        """
        Fit the Lee-Carter model using SVD.
        """
        self.ages = ages
        self.years = years

        log_mx = np.log(mx_df.values)
        self.a_x = np.mean(log_mx, axis=1)
        centered_log_mx = log_mx - self.a_x[:, np.newaxis]
        U, S, Vt = np.linalg.svd(centered_log_mx, full_matrices=False)
        self.b_x = U[:, 0]
        self.k_t = S[0] * Vt[0, :]

        b_sum = np.sum(self.b_x)
        self.b_x = self.b_x / b_sum
        self.k_t = self.k_t * b_sum

        k_mean = np.mean(self.k_t)
        self.k_t = self.k_t - k_mean
        self.a_x = self.a_x + self.b_x * k_mean

        k_t_diff = np.diff(self.k_t)
        self.drift = np.mean(k_t_diff)
        self.volatility = np.std(k_t_diff)
        
        print(f"Lee-Carter model fitted successfully.")
        print(f"  Ages: {int(ages.min())} to {int(ages.max())}")
        print(f"  Years: {int(years.min())} to {int(years.max())}")
        print(f"  Mortality trend drift: {self.drift:.6f}")
        print(f"  Mortality trend volatility: {self.volatility:.6f}")
        
    def predict_k_t(self, n_years_ahead, n_paths=1, seed=None):
        """
        Forecast k_t using random walk with drift.
        """
        rng_local = np.random.default_rng(seed)
        k_t_last = self.k_t[-1]
        
        k_t_future = np.zeros((n_paths, n_years_ahead))
        
        for path in range(n_paths):
            k_current = k_t_last
            for t in range(n_years_ahead):
                # k_{t+1} = k_t + drift + volatility * epsilon
                epsilon = rng_local.normal()
                k_current = k_current + self.drift + self.volatility * epsilon
                k_t_future[path, t] = k_current
        return k_t_future
    
    def forecast_mortality(self, n_years_ahead, n_paths=1, seed=None):
        """
        Forecast mortality rates m_x,t for future years.
        """
        k_t_future = self.predict_k_t(n_years_ahead, n_paths, seed)

        n_ages = len(self.ages)
        mx_future = np.zeros((n_paths, n_ages, n_years_ahead))
        
        for path in range(n_paths):
            for t in range(n_years_ahead):
                log_mx = self.a_x + self.b_x * k_t_future[path, t]
                mx_future[path, :, t] = np.exp(log_mx)        
        return mx_future
    
    def get_q_from_m(self, mx):
        """
        Convert central death rate (m_x) to probability of death (q_x).
        Uses the standard actuarial approximation: q_x ≈ m_x / (1 + 0.5 * m_x)
        """
        return mx / (1 + 0.5 * mx)

if mx_df is not None:
    lc_model = LeeCarter()
    lc_model.fit(mx_df, ages_hmd, years_hmd)
else:
    print("Error: mx_df not loaded. Check the HMD file path.")
    lc_model = None

In [ ]:
# -------------------------
# 4) VAParams class - contract parameters
# -------------------------
class VAParams:
    def __init__(self, dt=1.0, lc_model=None, use_stochastic_rates=True, use_stochastic_mortality=True):
        self.T = 25                    # years to maturity
        self.dt = dt
        self.T_steps = int(self.T / self.dt)
        
        self.start_age = 50            # Age of policyholder at purchase
        self.ann_max_age = 150         # Annuity pays up to this age
        
        # Calculate max steps needed for mortality (from start_age to ann_max_age)
        self.max_mortality_years = self.ann_max_age - self.start_age
        self.max_mortality_steps = int(self.max_mortality_years / self.dt) + 10
        
        self.r = 0.04                  # risk-free rate
        self.sigma = 0.15              # initial volatility of underlying fund S
        self.S0 = 100.0                # initial fund price (or account unit)
        self.P = 10000.0               # single premium (initial account)
        self.phi = 0.015               # continuous guarantee fee

        # --- Heston Stochastic Volatility Parameters ---
        self.v0 = self.sigma**2        # initial variance
        self.kappa_v = 1.5             # speed of variance mean reversion
        self.theta_v = self.sigma**2   # long-term variance
        self.sigma_v = 0.3             # volatility of volatility
        self.rho = -0.7                # correlation between asset and variance processes
        self.rho_Sr = 0.0              # equity-rate correlation

        # --- Mean-Reversion Parameters ---
        self.kappa = 0.5               # speed of reversion
        self.theta = np.log(self.S0 * 5)   # long-term mean of log-price

        # --- Jump-Diffusion Parameters ---
        self.lam = 0.05                 # jump intensity
        self.jump_mean = -0.075          # mean of log jump size
        self.jump_std = 0.15            # std dev of log jump size
        # risk-neutral compensator for jumps
        self.jump_compensator = self.lam * (np.exp(self.jump_mean + 0.5 * self.jump_std**2) - 1)

        # --- CIR Bond Model Parameters ---
        self.r0 = self.r               # initial interest rate
        self.kappa_r = 0.3             # speed of mean reversion for interest rates
        self.theta_r = self.r          # long-term mean interest rate
        self.sigma_r = 0.05            # volatility of interest rate
        self.bond_duration = 5.0       # duration for bond pricing (years)
        self.use_stochastic_rates = use_stochastic_rates
        
        # --- Portfolio Allocation Parameters ---
        self.w_stock = 0.8             # stock weight
        self.w_bond = 0.2              # bond weight
        self.rebalance_freq = 1.0      # rebalancing frequency in years
        self.rho_sb = -0.15             # correlation between stock and bond returns

        # --- GMIB Parameters ---
        self.ann_rate_current = 0.08
        self.ann_rate_guar = 0.08

        # --- Guarantee Base Roll-up Rates ---
        self.rollup_GMAB = 0.0
        self.rollup_GMIB = 0.0
        self.rollup_GMDB = 0.0

        # GMWB parameters
        self.gmw_total = self.P * 1.5
        self.gmw_annual_frac = 0.10   # fraction of initial premium per year
        self.gmw_step_frac = self.gmw_annual_frac * self.dt # Withdrawal allowance per step
        self.stepup_factors = [1.0] * (self.T_steps + 1)

        # --- Contract Flags ---
        self.has_GMAB = False
        self.has_GMIB = False
        self.has_GMWB = True
        self.has_GMDB = True

        # --- Stochastic Mortality Parameters ---
        self.lc_model = lc_model
        self.use_stochastic_mortality = use_stochastic_mortality
        if lc_model is None:
            raise ValueError("Lee-Carter model is required. Please fit the model first.")
        
        self.q_reference = self._generate_reference_mortality()
    
    def _generate_reference_mortality(self):
        """Generate a reference mortality path using expected mortality from Lee-Carter model."""
        q_ref = np.zeros(self.max_mortality_steps)
        
        k_t_last = self.lc_model.k_t[-1]
        
        for t_step in range(self.max_mortality_steps):
            k_forecast = k_t_last + self.lc_model.drift * (t_step + 1)
            current_year = int(t_step * self.dt)
            current_age = self.start_age + current_year
            age_to_lookup = min(current_age, self.lc_model.ages.max())
            age_idx = np.argmin(np.abs(self.lc_model.ages - age_to_lookup))
            log_mx = self.lc_model.a_x[age_idx] + self.lc_model.b_x[age_idx] * k_forecast
            mx = np.exp(log_mx)
            q_ref[t_step] = self.lc_model.get_q_from_m(mx)
        
        return q_ref

params = VAParams(dt=1, lc_model=lc_model)

In [ ]:
# -------------------------
# 5) Simulate Market Paths
# -------------------------
def simulate_cir_paths(params, n_paths, seed=None, n_steps=None, Z_r=None):
    """
    Simulates CIR short-rate paths with full truncation.
    """
    dt = params.dt
    T_steps = params.T_steps if n_steps is None else int(n_steps)
    if Z_r is None:
        rng_local = np.random.default_rng(seed)
    
    r_paths = np.empty((n_paths, T_steps + 1))
    r_paths[:, 0] = params.r0
    
    for t in range(T_steps):
        r_t = np.maximum(r_paths[:, t], 0)
        sqrt_r_t = np.sqrt(r_t)
        z_t = Z_r[:, t] if Z_r is not None else rng_local.normal(size=n_paths)
        dr = params.kappa_r * (params.theta_r - r_t) * dt + params.sigma_r * sqrt_r_t * np.sqrt(dt) * z_t
        r_paths[:, t + 1] = np.maximum(r_t + dr, 0)
    
    return r_paths

def simulate_k_paths(params, n_paths, seed=None, n_steps=None):
    """
    Simulates Lee-Carter k_t as a random walk with drift.
    """
    rng_local = np.random.default_rng(seed)
    dt = params.dt
    T_steps = params.max_mortality_steps if n_steps is None else int(n_steps)

    k0 = float(params.lc_model.k_t[-1])
    mu_k = float(params.lc_model.drift)
    sigma_k = float(params.lc_model.volatility) if params.use_stochastic_mortality else 0.0

    k_paths = np.empty((n_paths, T_steps + 1))
    k_paths[:, 0] = k0
    for t in range(T_steps):
        dk = mu_k * dt + sigma_k * np.sqrt(dt) * rng_local.normal(size=n_paths)
        k_paths[:, t + 1] = k_paths[:, t] + dk
    return k_paths

def _lc_q_from_k(k_vals, age_vals, params):
    """Vectorized q_x from Lee-Carter given k_t and age arrays (numpy)."""
    ages = params.lc_model.ages
    age_idx = np.abs(ages.reshape(1, -1) - age_vals.reshape(-1, 1)).argmin(axis=1)
    a_x = params.lc_model.a_x[age_idx]
    b_x = params.lc_model.b_x[age_idx]
    log_mx = a_x + b_x * k_vals
    mx = np.exp(log_mx)
    return params.lc_model.get_q_from_m(mx)

def simulate_paths(params, n_paths, seed=None):
    """
    Simulates stock price paths with stochastic volatility and mortality:
    1. Heston Stochastic Volatility + Merton Jumps under risk-neutral drift
    2. CIR short-rate model
    3. Lee-Carter Stochastic Mortality
    """
    rng_local = np.random.default_rng(seed)

    S0, T_steps, dt = params.S0, params.T_steps, params.dt
    v0, kappa_v, theta_v, sigma_v, rho = params.v0, params.kappa_v, params.theta_v, params.sigma_v, params.rho
    lam, jump_mean, jump_std, jump_compensator = params.lam, params.jump_mean, params.jump_std, params.jump_compensator
    rho_Sr = getattr(params, 'rho_Sr', 0.0)

    log_S = np.empty((n_paths, T_steps + 1))
    v = np.empty((n_paths, T_steps + 1))
    
    log_S[:, 0] = np.log(S0)
    v[:, 0] = v0

    Z_v = rng_local.normal(size=(n_paths, T_steps))
    Z_s_uncorr = rng_local.normal(size=(n_paths, T_steps))
    Z_s = rho * Z_v + np.sqrt(1 - rho**2) * Z_s_uncorr

    Z_r_indep = rng_local.normal(size=(n_paths, T_steps))
    if abs(rho_Sr) > 1e-10:
        Z_r = rho_Sr * Z_s_uncorr + np.sqrt(max(1.0 - rho_Sr**2, 0.0)) * Z_r_indep
    else:
        Z_r = Z_r_indep

    if params.use_stochastic_rates:
        r_paths = simulate_cir_paths(params, n_paths, Z_r=Z_r)
    else:
        r_paths = np.full((n_paths, T_steps + 1), params.r)

    for t in range(T_steps):
        v_t = np.maximum(v[:, t], 0)
        sqrt_v_t = np.sqrt(v_t)

        v_drift = kappa_v * (theta_v - v_t) * dt
        v_diffusion = sigma_v * sqrt_v_t * np.sqrt(dt) * Z_v[:, t]
        v[:, t+1] = v_t + v_drift + v_diffusion

        N = rng_local.poisson(lam * dt, size=n_paths)
        total_jumps = np.sum(N)
        jumps = rng_local.normal(loc=jump_mean, scale=jump_std, size=total_jumps)
        jump_sum = np.zeros(n_paths)
        jump_indices = np.repeat(np.arange(n_paths), N)
        np.add.at(jump_sum, jump_indices, jumps)

        r_t = r_paths[:, t]
        log_S_drift = (r_t - 0.5 * v_t - jump_compensator) * dt
        log_S_diffusion = sqrt_v_t * np.sqrt(dt) * Z_s[:, t]
        log_S[:, t+1] = log_S[:, t] + log_S_drift + log_S_diffusion + jump_sum
    
    k_paths_long = simulate_k_paths(params, n_paths, seed=rng_local.integers(0, 2**31), n_steps=params.max_mortality_steps)
    q_paths = np.zeros((n_paths, params.max_mortality_steps))
    for t_step in range(params.max_mortality_steps):
        current_year = int(t_step * dt)
        current_age = params.start_age + current_year
        age_vals = np.full(n_paths, current_age, dtype=float)
        k_vals = k_paths_long[:, t_step]
        q_paths[:, t_step] = _lc_q_from_k(k_vals, age_vals, params)

    k_paths = k_paths_long[:, :T_steps + 1]

    return np.exp(log_S), v, q_paths, r_paths, k_paths

In [ ]:
# -------------------------
# 6) DP Method - PRICING
# -------------------------
from scipy.interpolate import RegularGridInterpolator

def annuity_pv(c, T_start_years, T_max, r, start_age, dt, q_path, path_start_year=0, r_path=None):
    """
    Calculates PV of a deferred annuity using continuous discounting (exp(-∫ r dt)).
    """
    current_age = start_age + path_start_year * dt
    payment_years_count = int(T_max - current_age)
    if payment_years_count <= 0:
        return 0.0

    steps_per_year = int(1/dt)
    
    pv = 0.0
    prob_survival = 1.0
    
    r_cum = None
    if r_path is not None:
        r_path = np.asarray(r_path)
        r_cum = np.cumsum(r_path * dt)
    
    for t_year in range(1, payment_years_count + 1):
    
        rel_year_start_idx = (t_year - 1) * steps_per_year
        year_idx = path_start_year + rel_year_start_idx
        
        if year_idx >= len(q_path):
            break
            
        q_annual = q_path[year_idx] 
        
        prob_survival *= (1 - q_annual)
        
        if r_cum is None:
            discount = np.exp(-r * t_year)
        else:
            end_idx = min(path_start_year + t_year * steps_per_year, len(r_path))
            if end_idx <= path_start_year:
                break
            if path_start_year == 0:
                integral = r_cum[end_idx - 1]
            else:
                integral = r_cum[end_idx - 1] - r_cum[path_start_year - 1]
            discount = np.exp(-integral)
        
        if t_year > T_start_years:
            pv += c * prob_survival * discount
            
    return pv

def _base_at_time(params, t_years, rollup):
    """Deterministic guarantee base with continuous roll-up."""
    return params.P * np.exp(rollup * t_years)

def _update_base_after_withdrawal(B, A_pre, withdrawal, rollup, dt):
    """Full base dynamics: roll-up then pro-rata reduction by withdrawal."""
    B_roll = B * np.exp(rollup * dt)
    if withdrawal <= 0.0:
        return max(B_roll, 0.0)
    A_safe = max(A_pre, 1e-8)
    B_new = B_roll * (1.0 - withdrawal / A_safe)
    return max(B_new, 0.0)

def _build_q_paths_mc(params, n_paths, seed=1234):
    """
    Build stochastic mortality paths (q_paths) for DP using Lee-Carter k_t SDE.
    Returns q_paths with shape (n_paths, max_mortality_steps).
    """
    rng_local = np.random.default_rng(seed)
    k_paths = simulate_k_paths(params, n_paths, seed=rng_local.integers(0, 2**31), n_steps=params.max_mortality_steps)
    q_paths = np.zeros((n_paths, params.max_mortality_steps))
    for t_step in range(params.max_mortality_steps):
        current_year = int(t_step * params.dt)
        current_age = params.start_age + current_year
        age_vals = np.full(n_paths, current_age, dtype=float)
        k_vals = k_paths[:, t_step]
        q_paths[:, t_step] = _lc_q_from_k(k_vals, age_vals, params)
    return q_paths, k_paths

def _build_r_paths_mc(params, n_paths, seed=4321):
    """
    Build stochastic short-rate paths for DP (CIR).
    Returns r_paths with shape (n_paths, max_mortality_steps+1).
    """
    if params.use_stochastic_rates:
        return simulate_cir_paths(params, n_paths, seed=seed, n_steps=params.max_mortality_steps)
    return np.full((n_paths, params.max_mortality_steps + 1), params.r)

def _annuity_factor_from_qpaths(q_paths, params, path_start_year, r_paths=None):
    """
    Monte Carlo estimate of annuity factor (PV of $1 annual payment)
    using stochastic mortality paths. Used for GMIB annuity valuation in DP.
    """
    factors = []
    for i in range(q_paths.shape[0]):
        factors.append(
            annuity_pv(
                c=1.0,
                T_start_years=0,
                T_max=params.ann_max_age,
                r=params.r,
                start_age=params.start_age,
                dt=params.dt,
                q_path=q_paths[i],
                path_start_year=path_start_year,
                r_path=None if r_paths is None else r_paths[i]
            )
        )
    return float(np.mean(factors)) if len(factors) > 0 else 0.0

def dp_solver(
    params,
    n_A=20,
    n_G=20,
    n_V=10,
    n_B=10,
    n_R=5,
    n_K=5,
    n_return_samples=250,
    n_mortality_samples=None
):

    T_steps = params.T_steps
    dt = params.dt
    rng_local = np.random.default_rng(12345)
    
    expected_log_S = np.zeros(T_steps + 1)
    expected_log_S[0] = np.log(params.S0)
    for t in range(T_steps):
        expected_log_S[t+1] = params.theta + (expected_log_S[t] - params.theta) * np.exp(-params.kappa * dt)

    if n_mortality_samples is None:
        n_mortality_samples = n_return_samples
    q_paths_mc, _k_paths_mc = _build_q_paths_mc(params, n_mortality_samples, seed=2468)
    r_paths_mc_mort = _build_r_paths_mc(params, n_mortality_samples, seed=8642)

    annuity_factor_cache = {}
    if params.has_GMIB:
        for t_step in range(0, T_steps + 1):
            is_anniversary = (t_step > 0) and (t_step % int(1/dt) == 0) if dt < 1 else (t_step > 0)
            if is_anniversary or t_step == T_steps:
                annuity_factor_cache[t_step] = _annuity_factor_from_qpaths(
                    q_paths_mc,
                    params,
                    path_start_year=t_step,
                    r_paths=r_paths_mc_mort
                )

    A_grid = np.linspace(0.0, params.P * 3.0, n_A)
    if getattr(params, "sigma_v", 0.0) > 0 or getattr(params, "theta_v", 0.0) > 0:
        V_grid = np.linspace(0.0, params.theta_v * 3.0, n_V)
    else:
        V_grid = np.array([float(getattr(params, "v0", 0.0))]) 

    if params.use_stochastic_rates:
        r_min = 0.0
        r_max = max(params.theta_r + 4.0 * params.sigma_r, params.r * 2.0)
        r_grid = np.linspace(r_min, r_max, n_R)
    else:
        r_grid = np.array([params.r])

    k0 = float(params.lc_model.k_t[-1])
    mu_k = float(params.lc_model.drift)
    sigma_k = float(params.lc_model.volatility) if params.use_stochastic_mortality else 0.0
    if n_K <= 1:
        k_grid = np.array([k0])
    else:
        k_min = k0 + mu_k * 0.0 - 4.0 * sigma_k * np.sqrt(params.T)
        k_max = k0 + mu_k * params.T + 4.0 * sigma_k * np.sqrt(params.T)
        k_grid = np.linspace(k_min, k_max, n_K)

    def _q_step_from_k(k_val, t_step):
        current_year = int(t_step * dt)
        current_age = params.start_age + current_year
        q_annual = _lc_q_from_k(np.array([k_val]), np.array([current_age]), params)[0]
        return 1 - (1 - q_annual) ** dt

    def get_next_states(current_log_S, current_v, current_r, current_k, n_samples):
        kappa_v, theta_v, sigma_v, rho = params.kappa_v, params.theta_v, params.sigma_v, params.rho
        lam, jump_mean, jump_std, jump_compensator = params.lam, params.jump_mean, params.jump_std, params.jump_compensator

        Z_v = rng_local.normal(size=n_samples)
        Z_s_uncorr = rng_local.normal(size=n_samples)
        Z_s = rho * Z_v + np.sqrt(1 - rho**2) * Z_s_uncorr

        v_t = np.maximum(current_v, 0)
        sqrt_v_t = np.sqrt(v_t)
        v_drift = kappa_v * (theta_v - v_t) * dt
        v_diffusion = sigma_v * sqrt_v_t * np.sqrt(dt) * Z_v
        v_next = v_t + v_drift + v_diffusion
        v_next = np.maximum(v_next, 0)

        N = rng_local.poisson(lam * dt, size=n_samples)
        jumps = rng_local.normal(loc=jump_mean, scale=jump_std, size=np.sum(N))
        jump_sum = np.zeros(n_samples)
        jump_indices = np.repeat(np.arange(n_samples), N)
        np.add.at(jump_sum, jump_indices, jumps)

        if params.use_stochastic_rates:
            r_t = max(current_r, 0.0)
            dr = params.kappa_r * (params.theta_r - r_t) * dt + params.sigma_r * np.sqrt(r_t) * np.sqrt(dt) * rng_local.normal(size=n_samples)
            r_next = np.maximum(r_t + dr, 0.0)
        else:
            r_next = np.full(n_samples, params.r)

        if params.use_stochastic_mortality:
            dk = mu_k * dt + sigma_k * np.sqrt(dt) * rng_local.normal(size=n_samples)
            k_next = current_k + dk
        else:
            dk = mu_k * dt
            k_next = np.full(n_samples, current_k + dk)

        r_step_samples = np.full(n_samples, current_r)
        log_S_drift = (r_step_samples - 0.5 * v_t - jump_compensator) * dt
        log_S_diffusion = sqrt_v_t * np.sqrt(dt) * Z_s
        
        asset_returns = np.exp(log_S_drift + log_S_diffusion + jump_sum) * (1 - params.phi * dt)
        
        return asset_returns, v_next, r_next, k_next

    if params.has_GMWB:
        G_W_grid = np.linspace(0.0, params.gmw_total, n_G)
        
        if params.has_GMDB:
            B_D_grid = np.linspace(0.0, params.P * 3.0, n_B)
            V = np.zeros((T_steps + 1, n_A, n_G, n_B, n_V, len(r_grid), len(k_grid)))
            policy = np.zeros((T_steps, n_A, n_G, n_B, n_V, len(r_grid), len(k_grid)))

            A_mesh, G_W_mesh, B_D_mesh, V_mesh, R_mesh, K_mesh = np.meshgrid(
                A_grid, G_W_grid, B_D_grid, V_grid, r_grid, k_grid, indexing='ij'
            )
            V[T_steps, :, :, :, :, :, :] = A_mesh + G_W_mesh

            for t_step in tqdm(range(T_steps - 1, -1, -1), desc="DP backward induction (GMWB+GMDB)", unit="step", total=T_steps):
                interp_V_next = RegularGridInterpolator(
                    (A_grid, G_W_grid, B_D_grid, V_grid, r_grid, k_grid),
                    V[t_step + 1, :, :, :, :, :, :],
                    bounds_error=False,
                    fill_value=None
                )
                
                for iA, A in enumerate(A_grid):
                    for iG, G_W in enumerate(G_W_grid):
                        for iB, B_D in enumerate(B_D_grid):
                            for iV, v_market in enumerate(V_grid):
                                for iR, r_market in enumerate(r_grid):
                                    for iK, k_market in enumerate(k_grid):
                                        if G_W <= 1e-6:
                                            V[t_step, iA, iG, iB, iV, iR, iK] = A * np.exp(-r_market * (T_steps - t_step) * dt)
                                            continue

                                        q_step = _q_step_from_k(k_market, t_step)
                                        discount = np.exp(-r_market * dt)

                                        current_log_S_approx = expected_log_S[t_step]
                                        eps_returns, v_next, r_next, k_next = get_next_states(
                                            current_log_S_approx,
                                            v_market,
                                            r_market,
                                            k_market,
                                            n_return_samples
                                        )

                                        best_val_for_state = -1e18
                                        best_action_for_state = 0.0
                                        
                                        step_withdrawal_allowance = params.gmw_step_frac * params.P
                                        
                                        for act_frac in np.linspace(0, 1, 150):
                                            withdrawal = min(act_frac * step_withdrawal_allowance, G_W)
                                            paid_from_account = min(A, withdrawal)
                                            A_after_withdrawal = A - paid_from_account
                                            G_W_after_withdrawal = G_W - withdrawal
                                            B_D_after = _update_base_after_withdrawal(B_D, A, withdrawal, params.rollup_GMDB, dt)

                                            A_next = A_after_withdrawal * eps_returns
                                            
                                            points_to_interp = np.stack([
                                                A_next,
                                                np.full_like(A_next, G_W_after_withdrawal),
                                                np.full_like(A_next, B_D_after),
                                                v_next,
                                                r_next,
                                                k_next
                                            ], axis=-1)
                                            
                                            continuation_samples = interp_V_next(points_to_interp)
                                            
                                            death_benefit_value = max(B_D_after, A_after_withdrawal) + G_W_after_withdrawal
                                            expected_value_at_t_plus_1 = (1 - q_step) * continuation_samples + q_step * death_benefit_value
                                            current_value = withdrawal + discount * np.mean(expected_value_at_t_plus_1)

                                            if current_value > best_val_for_state:
                                                best_val_for_state, best_action_for_state = current_value, withdrawal
                                                
                                        V[t_step, iA, iG, iB, iV, iR, iK] = best_val_for_state
                                        policy[t_step, iA, iG, iB, iV, iR, iK] = best_action_for_state
            
            return {'A_grid': A_grid, 'G_grid': G_W_grid, 'B_grid': B_D_grid, 'V_grid': V_grid, 'R_grid': r_grid, 'K_grid': k_grid, 'V': V, 'policy': policy}

        V = np.zeros((T_steps + 1, n_A, n_G, n_V, len(r_grid), len(k_grid)))
        policy = np.zeros((T_steps, n_A, n_G, n_V, len(r_grid), len(k_grid)))

        A_mesh, G_W_mesh, V_mesh, R_mesh, K_mesh = np.meshgrid(A_grid, G_W_grid, V_grid, r_grid, k_grid, indexing='ij')
        V[T_steps, :, :, :, :, :] = A_mesh + G_W_mesh

        for t_step in tqdm(range(T_steps - 1, -1, -1), desc="DP backward induction (GMWB)", unit="step", total=T_steps):
            interp_V_next = RegularGridInterpolator(
                (A_grid, G_W_grid, V_grid, r_grid, k_grid),
                V[t_step + 1, :, :, :, :, :],
                bounds_error=False,
                fill_value=None
            )
            
            for iA, A in enumerate(A_grid):
                for iG, G_W in enumerate(G_W_grid):
                    for iV, v_market in enumerate(V_grid):
                        for iR, r_market in enumerate(r_grid):
                            for iK, k_market in enumerate(k_grid):
                                if G_W <= 1e-6:
                                    V[t_step, iA, iG, iV, iR, iK] = A * np.exp(-r_market * (T_steps - t_step) * dt)
                                    continue

                                q_step = _q_step_from_k(k_market, t_step)
                                discount = np.exp(-r_market * dt)

                                current_log_S_approx = expected_log_S[t_step]
                                eps_returns, v_next, r_next, k_next = get_next_states(
                                    current_log_S_approx,
                                    v_market,
                                    r_market,
                                    k_market,
                                    n_return_samples
                                )

                                best_val_for_state = -1e18
                                best_action_for_state = 0.0
                                
                                step_withdrawal_allowance = params.gmw_step_frac * params.P
                                
                                for act_frac in np.linspace(0, 1, 150):
                                    withdrawal = min(act_frac * step_withdrawal_allowance, G_W)
                                    paid_from_account = min(A, withdrawal)
                                    A_after_withdrawal = A - paid_from_account
                                    G_W_after_withdrawal = G_W - withdrawal

                                    A_next = A_after_withdrawal * eps_returns
                                    
                                    points_to_interp = np.stack([
                                        A_next,
                                        np.full_like(A_next, G_W_after_withdrawal),
                                        v_next,
                                        r_next,
                                        k_next
                                    ], axis=-1)
                                    
                                    continuation_samples = interp_V_next(points_to_interp)
                                    
                                    death_benefit_value = (A_after_withdrawal + G_W_after_withdrawal) if params.has_GMDB else A_after_withdrawal
                                    expected_value_at_t_plus_1 = (1 - q_step) * continuation_samples + q_step * death_benefit_value
                                    current_value = withdrawal + discount * np.mean(expected_value_at_t_plus_1)

                                    if current_value > best_val_for_state:
                                        best_val_for_state, best_action_for_state = current_value, withdrawal
                                        
                                V[t_step, iA, iG, iV, iR, iK] = best_val_for_state
                                policy[t_step, iA, iG, iV, iR, iK] = best_action_for_state
        
        return {'A_grid': A_grid, 'G_grid': G_W_grid, 'V_grid': V_grid, 'R_grid': r_grid, 'K_grid': k_grid, 'V': V, 'policy': policy}

    else:
        G_grid = np.array([params.P]) 
        V = np.zeros((T_steps + 1, n_A, n_V, len(r_grid), len(k_grid)))
        policy = np.zeros((T_steps, n_A, n_V, len(r_grid), len(k_grid)))

        A_mesh, V_mesh, R_mesh, K_mesh = np.meshgrid(A_grid, V_grid, r_grid, k_grid, indexing='ij')
        v_continue = A_mesh
        if params.has_GMAB:
            base_A_T = _base_at_time(params, params.T, params.rollup_GMAB)
            v_continue = np.maximum(v_continue, base_A_T)
        final_val = v_continue
        if params.has_GMIB:
            ann_factor_T = annuity_factor_cache.get(T_steps, 0.0)
            base_I_T = _base_at_time(params, params.T, params.rollup_GMIB)
            pv_ann_account = A_mesh * params.ann_rate_current * ann_factor_T
            pv_ann_guar = base_I_T * params.ann_rate_guar * ann_factor_T
            v_annuitize = np.maximum(pv_ann_account, pv_ann_guar)
            final_val = np.maximum(v_continue, v_annuitize)
        V[T_steps, :, :, :, :] = final_val

        for t_step in tqdm(range(T_steps - 1, -1, -1), desc="DP backward induction", unit="step", total=T_steps):
            interp_V_next = RegularGridInterpolator(
                (A_grid, V_grid, r_grid, k_grid),
                V[t_step + 1, :, :, :, :],
                bounds_error=False,
                fill_value=None
            )

            for iA, A in enumerate(A_grid):
                for iV, v_market in enumerate(V_grid):
                    for iR, r_market in enumerate(r_grid):
                        for iK, k_market in enumerate(k_grid):
                            q_step = _q_step_from_k(k_market, t_step)
                            discount = np.exp(-r_market * dt)

                            current_log_S_approx = expected_log_S[t_step]
                            eps_returns, v_next, r_next, k_next = get_next_states(
                                current_log_S_approx,
                                v_market,
                                r_market,
                                k_market,
                                n_return_samples
                            )

                            if params.has_GMIB:
                                A_after_withdrawal = A
                                points_to_interp = np.stack([A_after_withdrawal * eps_returns, v_next, r_next, k_next], axis=-1)
                                continuation_samples = interp_V_next(points_to_interp)
                                
                                base_D_t = _base_at_time(params, t_step * dt, params.rollup_GMDB) if params.has_GMDB else 0.0
                                death_benefit_value = max(base_D_t, A_after_withdrawal) if params.has_GMDB else A_after_withdrawal
                                expected_value_at_t_plus_1 = (1 - q_step) * continuation_samples + q_step * death_benefit_value
                                v_continue = np.mean(discount * expected_value_at_t_plus_1)

                                is_anniversary = (t_step > 0) and (t_step % int(1/dt) == 0) if dt < 1 else (t_step > 0)

                                if is_anniversary:
                                    ann_factor = annuity_factor_cache.get(t_step, 0.0)
                                    base_I_t = _base_at_time(params, t_step * dt, params.rollup_GMIB)
                                    pv_ann_account = A * params.ann_rate_current * ann_factor
                                    pv_ann_guar = base_I_t * params.ann_rate_guar * ann_factor
                                    
                                    v_annuitize = max(pv_ann_account, pv_ann_guar)
                                    
                                    if v_annuitize > v_continue:
                                        V[t_step, iA, iV, iR, iK] = v_annuitize
                                        policy[t_step, iA, iV, iR, iK] = 1 # Annuitize
                                    else:
                                        V[t_step, iA, iV, iR, iK] = v_continue
                                        policy[t_step, iA, iV, iR, iK] = 0 # Continue
                                else:
                                    V[t_step, iA, iV, iR, iK] = v_continue
                                    policy[t_step, iA, iV, iR, iK] = 0
                            else: 
                                if params.has_GMAB or (params.has_GMDB and not params.has_GMWB and not params.has_GMIB):
                                    A_after_withdrawal = A

                                    points_to_interp = np.stack([A_after_withdrawal * eps_returns, v_next, r_next, k_next], axis=-1)
                                    continuation_samples = interp_V_next(points_to_interp)

                                    base_D_t = _base_at_time(params, t_step * dt, params.rollup_GMDB) if params.has_GMDB else 0.0
                                    death_benefit_value = max(base_D_t, A_after_withdrawal) if params.has_GMDB else A_after_withdrawal
                                    expected_value_at_t_plus_1 = (1 - q_step) * continuation_samples + q_step * death_benefit_value
                                    current_value = np.mean(discount * expected_value_at_t_plus_1)

                                    V[t_step, iA, iV, iR, iK], policy[t_step, iA, iV, iR, iK] = current_value, 0.0
                                else:
                                    best_val_for_state = -1e18
                                    best_action_for_state = 0.0
                                    for act_frac in np.linspace(0, 1, 150):
                                        withdrawal = min(act_frac * A, A)
                                        A_after_withdrawal = A - withdrawal

                                        points_to_interp = np.stack([A_after_withdrawal * eps_returns, v_next, r_next, k_next], axis=-1)
                                        continuation_samples = interp_V_next(points_to_interp)

                                        base_D_t = _base_at_time(params, t_step * dt, params.rollup_GMDB) if params.has_GMDB else 0.0
                                        death_benefit_value = max(base_D_t, A_after_withdrawal) if params.has_GMDB else A_after_withdrawal
                                        expected_value_at_t_plus_1 = (1 - q_step) * continuation_samples + q_step * death_benefit_value
                                        current_value = withdrawal + np.mean(discount * expected_value_at_t_plus_1)
                                        if current_value > best_val_for_state:
                                            best_val_for_state, best_action_for_state = current_value, withdrawal
                                    V[t_step, iA, iV, iR, iK], policy[t_step, iA, iV, iR, iK] = best_val_for_state, best_action_for_state

        return {'A_grid': A_grid, 'G_grid': G_grid, 'V_grid': V_grid, 'R_grid': r_grid, 'K_grid': k_grid, 'V': V, 'policy': policy}

In [ ]:
# -------------------------
# 7) Han Method - PRICING
# -------------------------
def _base_at_time_scalar(params, t_years, rollup):
    return float(params.P * np.exp(rollup * t_years))

class HanPolicy:
    def __init__(self, params, hidden_units=64, learning_rate=1e-3, hidden_layers=2, dropout_rate=0.0, activation="tanh"):
        self.params = params
        self.T_steps = params.T_steps
        self.n_state = 7
        
        self.models = []
        for t in range(self.T_steps):
            inputs = Input(shape=(self.n_state,))
            x = inputs
            for _ in range(hidden_layers):
                x = layers.Dense(hidden_units, activation=activation)(x)
                if dropout_rate and dropout_rate > 0:
                    x = layers.Dropout(dropout_rate)(x)
            if params.has_GMIB:
                outputs = layers.Dense(1, activation=None)(x)
            else:
                outputs = layers.Dense(1, activation='softplus')(x)
            model = Model(inputs, outputs)
            self.models.append(model)
        self.opt = optimizers.Adam(learning_rate=learning_rate)

    def train(self, params, epochs=200, n_paths=200, verbose=True):
        all_vars = [v for m in self.models for v in m.trainable_variables]
        
        for ep in tqdm(range(epochs), desc="Han training", unit="epoch"):
            with tf.GradientTape() as tape:
                S_paths, V_paths, q_paths, r_paths, k_paths = simulate_paths(params, n_paths, seed=None)
                q_paths_tf = tf.convert_to_tensor(q_paths, dtype=tf.float32)
                r_paths_tf = tf.convert_to_tensor(r_paths, dtype=tf.float32)
                k_paths_tf = tf.convert_to_tensor(k_paths, dtype=tf.float32)
                
                if params.has_GMWB:
                    A = tf.fill([n_paths], float(params.P))
                    W = tf.fill([n_paths], float(params.gmw_total))
                    B_D = tf.fill([n_paths], float(params.P)) if params.has_GMDB else tf.zeros([n_paths])
                    
                    withdrawals_hist, A_after_withdrawal_hist, W_after_withdrawal_hist = [], [], []
                    B_D_after_withdrawal_hist = []

                    for t_step in range(1, params.T_steps + 1):
                        St = tf.convert_to_tensor(S_paths[:, t_step].astype(np.float32))
                        Vt = tf.convert_to_tensor(V_paths[:, t_step].astype(np.float32))
                        S_prev = tf.convert_to_tensor(S_paths[:, t_step-1].astype(np.float32))
                        r_t = tf.gather(r_paths_tf, t_step, axis=1)
                        k_t = tf.gather(k_paths_tf, t_step, axis=1)
                        t_years = tf.fill([n_paths], float(t_step * params.dt))

                        state = self._create_state_vector(t_years, A, W, B_D, r_t, Vt, k_t, params)
                        
                        withdrawal_pred = self.models[t_step-1](state)[:, 0]
                        
                        step_cap = params.gmw_step_frac * params.P
                        act_withdrawals = tf.minimum(withdrawal_pred, step_cap)
                        act_withdrawals = tf.minimum(act_withdrawals, W)
                        
                        withdrawals_hist.append(act_withdrawals)
                        
                        paid_from_account = tf.minimum(A, act_withdrawals)
                        A_after = A - paid_from_account
                        W_after = W - act_withdrawals
                        
                        A_after_withdrawal_hist.append(A_after)
                        W_after_withdrawal_hist.append(W_after)
                        
                        if params.has_GMDB:
                            A_safe = tf.maximum(A, 1e-8)
                            B_D = B_D * tf.exp(params.rollup_GMDB * params.dt) * (1.0 - act_withdrawals / A_safe)
                            B_D = tf.maximum(B_D, 0.0)
                            B_D_after_withdrawal_hist.append(B_D)
                        
                        scale = St / (S_prev + 1e-12)
                        A = A_after * scale * (1 - params.phi * params.dt)
                        W = W_after

                    V = A + W
                    for t_step in range(params.T_steps - 1, -1, -1):
                        q_annual = tf.gather(q_paths_tf, t_step, axis=1)
                        q_step = 1 - tf.pow(1 - q_annual, params.dt)
                        
                        r_step = tf.gather(r_paths_tf, t_step, axis=1)
                        discount_factor = tf.exp(-r_step * params.dt)
                        
                        if params.has_GMDB:
                            death_benefit = tf.maximum(A_after_withdrawal_hist[t_step], B_D_after_withdrawal_hist[t_step]) + W_after_withdrawal_hist[t_step]
                        else:
                            death_benefit = A_after_withdrawal_hist[t_step]
                        expected_future_val = (1 - q_step) * V + q_step * death_benefit
                        V = withdrawals_hist[t_step] + discount_factor * expected_future_val

                else:
                    A = tf.fill([n_paths], float(params.P))
                    A_after_withdrawal_hist = []
                    annuitize_decisions = []

                    withdrawals_hist = [tf.zeros_like(A)] * params.T_steps

                    for t_step in range(1, params.T_steps + 1):
                        St = tf.convert_to_tensor(S_paths[:, t_step].astype(np.float32))
                        Vt = tf.convert_to_tensor(V_paths[:, t_step].astype(np.float32))
                        S_prev = tf.convert_to_tensor(S_paths[:, t_step-1].astype(np.float32))
                        r_t = tf.gather(r_paths_tf, t_step, axis=1)
                        k_t = tf.gather(k_paths_tf, t_step, axis=1)
                        t_years = tf.fill([n_paths], float(t_step * params.dt))
                        B_D_base = _base_at_time_scalar(params, t_step * params.dt, params.rollup_GMDB) if params.has_GMDB else 0.0
                        B_D_vec = tf.fill([n_paths], float(B_D_base))
                        W_vec = tf.zeros([n_paths])
                        
                        if params.has_GMIB:
                            state = self._create_state_vector(t_years, A, W_vec, B_D_vec, r_t, Vt, k_t, params)
                            policy_output = self.models[t_step-1](state)
                            annuitize_decisions.append(tf.nn.sigmoid(policy_output[:, 0]))
                        else:
                            annuitize_decisions.append(tf.zeros(n_paths))
                        
                        scale = St / (S_prev + 1e-12)
                        A = A * scale * (1 - params.phi * params.dt)
                        
                        A_after_withdrawal_hist.append(A)
                    
                    v_continue_T = A
                    if params.has_GMAB:
                        base_A_T = _base_at_time_scalar(params, params.T, params.rollup_GMAB)
                        v_continue_T = tf.maximum(v_continue_T, float(base_A_T))
                    V = v_continue_T
                    
                    if params.has_GMIB:
                        current_age_T = tf.cast(params.start_age, tf.float32) + float(params.T)
                        base_I_T = _base_at_time_scalar(params, params.T, params.rollup_GMIB)
                        pv_ann_account_T = self._get_immediate_annuity_pv(A, params.ann_rate_current, current_age_T, params.T_steps, q_paths_tf, r_paths_tf, params)
                        pv_ann_guar_T = self._get_immediate_annuity_pv(float(base_I_T), params.ann_rate_guar, current_age_T, params.T_steps, q_paths_tf, r_paths_tf, params)
                        v_annuitize_T = tf.maximum(pv_ann_account_T, pv_ann_guar_T)
                        V = tf.maximum(v_continue_T, v_annuitize_T)
                    
                    for t_step in range(params.T_steps - 1, -1, -1):
                        if params.has_GMIB:
                            is_year_end = (t_step + 1) % int(1/params.dt) == 0 if params.dt < 1 else True
                            if is_year_end and (t_step + 1) < params.T_steps:
                                current_time_years = (t_step + 1) * params.dt
                                current_age_next = tf.cast(params.start_age, tf.float32) + current_time_years
                                base_I_current = _base_at_time_scalar(params, current_time_years, params.rollup_GMIB)
                                
                                A_at_t_plus_1 = A_after_withdrawal_hist[t_step] 
                                
                                pv_ann_account = self._get_immediate_annuity_pv(A_at_t_plus_1, params.ann_rate_current, current_age_next, t_step + 1, q_paths_tf, r_paths_tf, params)
                                pv_ann_guar = self._get_immediate_annuity_pv(float(base_I_current), params.ann_rate_guar, current_age_next, t_step + 1, q_paths_tf, r_paths_tf, params)
                                v_annuitize = tf.maximum(pv_ann_account, pv_ann_guar)
                                
                                decision = annuitize_decisions[t_step]
                                V = decision * v_annuitize + (1 - decision) * V

                        q_annual = tf.gather(q_paths_tf, t_step, axis=1)
                        q_step = 1 - tf.pow(1 - q_annual, params.dt)
                        
                        r_step = tf.gather(r_paths_tf, t_step, axis=1)
                        discount_factor = tf.exp(-r_step * params.dt)
                        
                        A_future = A_after_withdrawal_hist[t_step]
                        if params.has_GMDB:
                            base_D_t = _base_at_time_scalar(params, t_step * params.dt, params.rollup_GMDB)
                            base_D_tensor = tf.fill(tf.shape(A_future), float(base_D_t))
                            death_benefit = tf.maximum(base_D_tensor, A_future)
                        else:
                            death_benefit = A_future
                        
                        expected_future_val = (1 - q_step) * V + q_step * death_benefit
                        v_continue = withdrawals_hist[t_step] + discount_factor * expected_future_val
                        
                        V = v_continue
                
                loss = -tf.reduce_mean(V)
                if not tf.reduce_all(tf.math.is_finite(loss)):
                    if verbose:
                        tqdm.write(f"Han epoch {ep+1}/{epochs}, non-finite loss, skipping update.")
                    continue

            grads = tape.gradient(loss, all_vars)
            
            grads_vars = [(g, v) for g, v in zip(grads, all_vars) if g is not None]
            if not grads_vars:
                if verbose:
                    tqdm.write(f"Han epoch {ep+1}/{epochs}, no trainable policy for this rider, skipping gradient update.")
                continue
            
            grads_list, vars_list = zip(*grads_vars)
            grads_list, _ = tf.clip_by_global_norm(grads_list, 10.0)
            self.opt.apply_gradients(zip(grads_list, vars_list))
            
            if verbose and (ep % 25 == 0 or ep == epochs - 1):
                tqdm.write(f"  Han ep {ep+1:4d}/{epochs} | loss {loss.numpy():.4e} | value {-loss.numpy():.4e}")
        
        return -loss.numpy()

    def _create_state_vector(self, t_years, A, W, B_D, r_t, v_t, k_t, params):
        n_paths = tf.shape(A)[0]
        
        def _expand_or_fill(val, default_val):
            if val is None:
                return tf.fill([n_paths], float(default_val))
            val_t = tf.convert_to_tensor(val, dtype=tf.float32)
            if val_t.shape.rank == 0:
                return tf.fill([n_paths], val_t)
            return val_t
        
        t_val = _expand_or_fill(t_years, 0.0)
        W_val = _expand_or_fill(W, 0.0)
        B_D_val = _expand_or_fill(B_D, 0.0)
        r_val = _expand_or_fill(r_t, params.r)
        v_val = _expand_or_fill(v_t, params.v0)
        k_val = _expand_or_fill(k_t, params.lc_model.k_t[-1])
        return tf.stack([t_val, A, W_val, B_D_val, r_val, v_val, k_val], axis=1)
    
    @tf.function
    def _get_immediate_annuity_pv(self, capital, rate, current_age_tensor, current_step_idx, q_paths, r_paths, params):
        """
        Calculates PV of an annuity starting immediately
        """
        r = tf.cast(params.r, tf.float32)
        dt = tf.cast(params.dt, tf.float32)
        steps_per_year = tf.cast(1.0/dt, tf.int32)
        q_paths = tf.cast(q_paths, tf.float32)
        r_paths = tf.cast(r_paths, tf.float32)
        table_len = tf.shape(q_paths)[1]
        rate_table_len = tf.shape(r_paths)[1]
        
        r_cum = tf.cumsum(r_paths * dt, axis=1)
        r_cum_start = tf.gather(r_cum, current_step_idx, axis=1)
        
        max_age = tf.cast(params.ann_max_age, tf.float32)
        max_years = tf.cast(max_age - current_age_tensor, tf.int32)
        
        def pv_calc():
            payment_years = tf.range(1, max_years + 1)
            rel_indices = (payment_years - 1) * steps_per_year
            abs_indices = current_step_idx + rel_indices
            valid_mask = abs_indices < table_len
            abs_indices_clipped = tf.minimum(abs_indices, table_len - 1)
            
            q_values = tf.gather(q_paths, abs_indices_clipped, axis=1)
            q_values = tf.where(valid_mask, q_values, tf.ones_like(q_values))
            probs = tf.math.cumprod(1.0 - q_values, axis=1)
            
            abs_indices_r = tf.minimum(abs_indices, rate_table_len - 1)
            r_cum_at = tf.gather(r_cum, abs_indices_r, axis=1)
            discounts = tf.exp(-(r_cum_at - tf.expand_dims(r_cum_start, 1))) * tf.cast(valid_mask, tf.float32)
            factor = tf.reduce_sum(probs * discounts, axis=1)
            return tf.cast(capital, tf.float32) * tf.cast(rate, tf.float32) * factor
        
        return tf.cond(max_years > 0, pv_calc, lambda: tf.zeros_like(tf.cast(capital, tf.float32)))
    
def evaluate_han_policy_mc(han_policy, params, n_paths=2000, seed=0):
    """
    MC evaluator for the learned Han policy.
    """
    S_paths, V_paths, q_paths, r_paths, k_paths = simulate_paths(params, n_paths, seed=seed)
    rng_local = np.random.default_rng(seed + 10007)
    
    r_integral = np.cumsum(r_paths[:, :-1] * params.dt, axis=1)
    
    A = np.full(n_paths, params.P, dtype=float)
    W = np.full(n_paths, params.gmw_total, dtype=float) if params.has_GMWB else np.zeros(n_paths, dtype=float)
    B_D = np.full(n_paths, params.P, dtype=float) if params.has_GMDB else np.zeros(n_paths, dtype=float)
    alive = np.ones(n_paths, dtype=bool)
    payouts = np.zeros(n_paths, dtype=float)
    deterministic_eval = (not params.use_stochastic_rates) and (not params.use_stochastic_mortality) and (params.sigma_v == 0.0) and (params.lam == 0.0)
    
    for t_step in tqdm(range(1, params.T_steps + 1), desc="MC simulation", unit="step"):
        if not np.any(alive):
            break

        St = S_paths[:, t_step]
        Vt = V_paths[:, t_step]
        S_prev = S_paths[:, t_step-1]
        r_t = r_paths[:, t_step]
        k_t = k_paths[:, t_step]
        
        t_years = np.full(np.sum(alive), t_step * params.dt)
        W_state = W[alive] if params.has_GMWB else np.zeros(np.sum(alive))
        B_D_state = B_D[alive] if params.has_GMDB else np.zeros(np.sum(alive))
        state = han_policy._create_state_vector(
            tf.constant(t_years, dtype=tf.float32),
            tf.constant(A[alive], dtype=tf.float32),
            tf.constant(W_state, dtype=tf.float32),
            tf.constant(B_D_state, dtype=tf.float32),
            tf.constant(r_t[alive], dtype=tf.float32),
            tf.constant(Vt[alive], dtype=tf.float32),
            tf.constant(k_t[alive], dtype=tf.float32),
            params
        )
        policy_output = han_policy.models[t_step-1](state).numpy()
        if not np.all(np.isfinite(policy_output)):
            policy_output = np.nan_to_num(policy_output, nan=0.0, posinf=0.0, neginf=0.0)
        
        if params.has_GMAB or params.has_GMIB:
            withdrawals = np.zeros(np.sum(alive))
        else:
            withdrawals = policy_output[:, 0]
            
            if params.has_GMWB:
                step_cap = params.gmw_step_frac * params.P
                withdrawals = np.minimum(withdrawals, step_cap)
                withdrawals = np.minimum(withdrawals, W[alive])
            
            withdrawals = np.minimum(withdrawals, A[alive])
            withdrawals = np.maximum(withdrawals, 0.0)

        discount_t = np.exp(-r_integral[:, t_step-1])
        payouts[alive] += withdrawals * discount_t[alive]
        
        A[alive] -= withdrawals
        if params.has_GMWB:
            W[alive] -= withdrawals

        if params.has_GMDB and params.has_GMWB:
            A_before = A[alive] + withdrawals
            A_safe = np.maximum(A_before, 1e-8)
            B_D[alive] = B_D[alive] * np.exp(params.rollup_GMDB * params.dt) * (1.0 - withdrawals / A_safe)
            B_D[alive] = np.maximum(B_D[alive], 0.0)

        is_year_end = (t_step % int(1/params.dt) == 0) if params.dt < 1 else True
        if params.has_GMIB and is_year_end:
            logits = np.clip(policy_output[:, 0], -50.0, 50.0)
            annuitize_prob = 1 / (1 + np.exp(-logits))
            if deterministic_eval:
                does_annuitize = annuitize_prob >= 0.5
            else:
                does_annuitize = (rng_local.random(len(annuitize_prob)) < annuitize_prob)
            paths_annuitizing_now = np.where(alive)[0][does_annuitize]
            if len(paths_annuitizing_now) > 0:
                A_ann = A[paths_annuitizing_now]
                pv_ann_values = np.zeros(len(paths_annuitizing_now))
                
                current_age = params.start_age + t_step * params.dt
                base_I_current = params.P * np.exp(params.rollup_GMIB * (t_step * params.dt))
                
                for i, path_idx in enumerate(paths_annuitizing_now):
                    pv_ann_account = annuity_pv(c=A_ann[i] * params.ann_rate_current, T_start_years=0, T_max=params.ann_max_age, r=params.r, start_age=current_age, dt=params.dt, q_path=q_paths[path_idx], path_start_year=t_step, r_path=r_paths[path_idx])
                    pv_ann_guar = annuity_pv(c=base_I_current * params.ann_rate_guar, T_start_years=0, T_max=params.ann_max_age, r=params.r, start_age=current_age, dt=params.dt, q_path=q_paths[path_idx], path_start_year=t_step, r_path=r_paths[path_idx])
                    pv_ann_values[i] = max(pv_ann_account, pv_ann_guar)
                payouts[paths_annuitizing_now] += pv_ann_values * discount_t[paths_annuitizing_now]
                alive[paths_annuitizing_now] = False

        q_annual_all_paths = q_paths[:, t_step - 1]
        q_annual_alive = q_annual_all_paths[alive]
        q_step = 1 - (1 - q_annual_alive)**params.dt
        
        dies_this_year_indices = np.where(alive)[0][rng_local.random(np.sum(alive)) < q_step]
        if len(dies_this_year_indices) > 0:
            if params.has_GMWB and params.has_GMDB:
                death_payout = np.maximum(B_D[dies_this_year_indices], A[dies_this_year_indices]) + W[dies_this_year_indices]
            elif params.has_GMDB:
                base_D_current = params.P * np.exp(params.rollup_GMDB * (t_step * params.dt))
                death_payout = np.maximum(base_D_current, A[dies_this_year_indices])
            else:
                death_payout = A[dies_this_year_indices]
            payouts[dies_this_year_indices] += death_payout * discount_t[dies_this_year_indices]
            alive[dies_this_year_indices] = False

        if np.any(alive):
            scale = St[alive] / (S_prev[alive] + 1e-12)
            A[alive] *= scale * (1 - params.phi * params.dt)
            A[alive] = np.maximum(A[alive], 0.0)

    if np.any(alive):
        discount_T = np.exp(-r_integral[:, -1])
        A_final = A[alive]
        
        final_payoff = A_final
        if params.has_GMWB:
            final_payoff += W[alive]
        if params.has_GMAB: 
            base_A_T = params.P * np.exp(params.rollup_GMAB * params.T)
            final_payoff = np.maximum(final_payoff, base_A_T)
            
        if params.has_GMIB:
            final_payoff_values = np.zeros(np.sum(alive))
            alive_indices = np.where(alive)[0]
            current_age_T = params.start_age + params.T
            base_I_T = params.P * np.exp(params.rollup_GMIB * params.T)
            
            for i, path_idx in enumerate(alive_indices):
                pv_ann_account_T = annuity_pv(c=A_final[i] * params.ann_rate_current, T_start_years=0, T_max=params.ann_max_age, r=params.r, start_age=current_age_T, dt=params.dt, q_path=q_paths[path_idx], path_start_year=params.T_steps, r_path=r_paths[path_idx])
                pv_ann_guar_T = annuity_pv(c=base_I_T * params.ann_rate_guar, T_start_years=0, T_max=params.ann_max_age, r=params.r, start_age=current_age_T, dt=params.dt, q_path=q_paths[path_idx], path_start_year=params.T_steps, r_path=r_paths[path_idx])
                v_annuitize_T = max(pv_ann_account_T, pv_ann_guar_T)
                final_payoff_values[i] = max(final_payoff[i], v_annuitize_T)
            payouts[alive] += final_payoff_values * discount_T[alive]
        else:
            payouts[alive] += final_payoff * discount_T[alive]

    return np.mean(payouts)

In [ ]:
# -------------------------
# 8) PINN Method - PRICING
# -------------------------
import torch
import torch.nn as nn
import torch.autograd as autograd

torch.set_default_dtype(torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

class ValueNet(nn.Module):
    def __init__(self, dim=7, hidden=256, depth=3, x_min=None, x_max=None):
        super().__init__()
        self.input_layer = nn.Linear(dim, hidden)
        self.hidden_layers = nn.ModuleList(
            [nn.Linear(hidden, hidden) for _ in range(depth - 1)]
        )
        self.skip = nn.Linear(dim, hidden)
        self.output_layer = nn.Linear(hidden, 1)
        nn.init.zeros_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
        if x_min is None or x_max is None:
            self.register_buffer("x_min", torch.zeros(dim))
            self.register_buffer("x_max", torch.ones(dim))
        else:
            self.register_buffer("x_min", x_min)
            self.register_buffer("x_max", x_max)

    def forward(self, x):
        x = x.to(self.x_min.dtype)
        x_scaled = (x - self.x_min) / (self.x_max - self.x_min + 1e-8)
        h = torch.tanh(self.input_layer(x_scaled))
        skip = self.skip(x_scaled)
        for layer in self.hidden_layers:
            h = torch.tanh(layer(h))
        h = h + skip
        return self.output_layer(h)

class DGMNet(nn.Module):
    """Deep Galerkin Method network (Sirignano & Spiliopoulos 2018)."""
    def __init__(self, dim=7, hidden=256, depth=3, x_min=None, x_max=None):
        super().__init__()
        self.depth = depth
        if x_min is None or x_max is None:
            self.register_buffer("x_min", torch.zeros(dim))
            self.register_buffer("x_max", torch.ones(dim))
        else:
            self.register_buffer("x_min", x_min)
            self.register_buffer("x_max", x_max)

        self.Sw = nn.Linear(dim, hidden)

        self.Uz  = nn.ModuleList([nn.Linear(dim, hidden) for _ in range(depth)])
        self.Wsz = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(depth)])
        self.Ug  = nn.ModuleList([nn.Linear(dim, hidden) for _ in range(depth)])
        self.Wsg = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(depth)])
        self.Ur  = nn.ModuleList([nn.Linear(dim, hidden) for _ in range(depth)])
        self.Wsr = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(depth)])
        self.Uh  = nn.ModuleList([nn.Linear(dim, hidden) for _ in range(depth)])
        self.Wsh = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(depth)])

        self.output_layer = nn.Linear(hidden, 1)
        nn.init.zeros_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, x):
        x = x.to(self.x_min.dtype)
        x_scaled = (x - self.x_min) / (self.x_max - self.x_min + 1e-8)
        S = torch.tanh(self.Sw(x_scaled))
        for i in range(self.depth):
            Z = torch.tanh(self.Uz[i](x_scaled) + self.Wsz[i](S))
            G = torch.tanh(self.Ug[i](x_scaled) + self.Wsg[i](S))
            R = torch.tanh(self.Ur[i](x_scaled) + self.Wsr[i](S))
            H = torch.tanh(self.Uh[i](x_scaled) + self.Wsh[i](S * R))
            S = (1.0 - G) * H + Z * S
        return self.output_layer(S)

def grad(y, x):
    return autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True
    )[0]

def hazard_rate_lc(t, k, params):
    """Hazard rate from Lee-Carter given current time and k_t state."""
    ages_min = int(params.lc_model.ages.min())
    ages_max = int(params.lc_model.ages.max())
    age = (params.start_age + t).squeeze(1)
    a_x = torch.tensor(params.lc_model.a_x, dtype=torch.float32, device=t.device)
    b_x = torch.tensor(params.lc_model.b_x, dtype=torch.float32, device=t.device)
    age_clamped = age.clamp(float(ages_min), float(ages_max) - 1e-6)
    age_lo = (age_clamped - ages_min).long()
    age_hi = (age_lo + 1).clamp(max=len(a_x) - 1)
    frac = (age_clamped - ages_min) - age_lo.float()
    a_sel = a_x[age_lo] * (1.0 - frac) + a_x[age_hi] * frac
    b_sel = b_x[age_lo] * (1.0 - frac) + b_x[age_hi] * frac
    log_mx = a_sel + b_sel * k.squeeze(1)
    mx = torch.exp(log_mx)
    q = mx / (1.0 + 0.5 * mx)
    q = torch.clamp(q, min=0.0, max=1.0 - 1e-8)
    return (-torch.log(1.0 - q)).unsqueeze(1)

def optimal_withdrawal(VA, VW, gamma_max, k=5.0):
    """Smooth withdrawal control based on HJB first-order condition."""
    if gamma_max <= 0:
        return torch.zeros_like(VA)
    indicator = 1.0 - VA - VW
    return gamma_max * torch.sigmoid(k * indicator)

def cir_zero_coupon_price(tau, r, params):
    """CIR zero-coupon bond price P(t, t+tau) under risk-neutral measure."""
    kappa = float(params.kappa_r)
    theta = float(params.theta_r)
    sigma = float(params.sigma_r)
    tau = torch.clamp(tau, min=0.0)
    gamma = torch.sqrt(torch.tensor(kappa**2 + 2.0 * sigma**2, device=r.device))
    exp_gt = torch.exp(gamma * tau)
    denom = (gamma + kappa) * (exp_gt - 1.0) + 2.0 * gamma
    B = 2.0 * (exp_gt - 1.0) / denom
    A = (2.0 * gamma * torch.exp((kappa + gamma) * tau * 0.5) / denom) ** (2.0 * kappa * theta / (sigma**2 + 1e-12))
    return A * torch.exp(-B * r)

def _annuity_factor_expected_k(t, k, r, params, use_cir=True):
    """Expected annuity factor using LC drift from current k_t state."""
    dt = float(params.dt)
    steps_per_year = int(1 / dt)
    max_years_total = int(params.ann_max_age - params.start_age)
    years = torch.arange(1, max_years_total + 1, device=r.device).float().unsqueeze(0)
    t_flat = t.squeeze(1)
    k_flat = k.squeeze(1)
    current_age = params.start_age + t_flat
    max_years = torch.clamp(params.ann_max_age - current_age, min=0.0)
    valid_mask = years <= max_years.unsqueeze(1)
    mu_k = float(params.lc_model.drift)
    k_future = k_flat.unsqueeze(1) + mu_k * years
    ages_min = int(params.lc_model.ages.min())
    ages_max = int(params.lc_model.ages.max())
    age_years = current_age.unsqueeze(1) + (years - 1.0)
    age_idx = torch.round(age_years).clamp(ages_min, ages_max) - ages_min
    age_idx = age_idx.long()
    a_x = torch.tensor(params.lc_model.a_x, dtype=torch.float32, device=r.device)
    b_x = torch.tensor(params.lc_model.b_x, dtype=torch.float32, device=r.device)
    a_sel = a_x[age_idx]
    b_sel = b_x[age_idx]
    log_mx = a_sel + b_sel * k_future
    mx = torch.exp(log_mx)
    q_vals = mx / (1.0 + 0.5 * mx)
    q_vals = torch.where(valid_mask, q_vals, torch.ones_like(q_vals))
    probs = torch.cumprod(1.0 - q_vals, dim=1)
    tau = years
    if use_cir:
        discounts = cir_zero_coupon_price(tau, r, params)
    else:
        discounts = torch.exp(-r * tau)
    factor = torch.sum(probs * discounts * valid_mask, dim=1, keepdim=True)
    return factor

def annuity_value(A, t, r, k, params, use_cir=True, ann_factors=None):
    if ann_factors is None:
        factor = _annuity_factor_expected_k(t, k, r, params, use_cir=use_cir)
    else:
        dt = float(params.dt)
        max_idx = ann_factors.shape[0] - 1
        idx = torch.round(t / dt).long().clamp(0, max_idx)
        factor = ann_factors[idx].reshape(-1, 1)
    ann_acc = A * params.ann_rate_current * factor
    base = torch.full_like(A, float(params.P)) * torch.exp(float(params.rollup_GMIB) * t)
    ann_guar = base * params.ann_rate_guar * factor
    return torch.maximum(ann_acc, ann_guar)

def terminal_payoff_contract(A, W, B_D, t, r, k, params, use_cir=True, ann_factors=None):
    if params.has_GMWB:
        return A + W
    if params.has_GMIB:
        return torch.maximum(
            A,
            annuity_value(A, t, r, k, params, use_cir=use_cir, ann_factors=ann_factors),
        )
    if params.has_GMAB:
        base = torch.full_like(A, float(params.P)) * torch.exp(float(params.rollup_GMAB) * t)
        return torch.maximum(A, base)
    return A

def death_benefit_contract(A, W, B_D, t, params):
    if not params.has_GMDB:
        return A
    if params.has_GMWB:
        base = B_D
        return torch.maximum(A, base) + W
    base = torch.full_like(A, float(params.P)) * torch.exp(float(params.rollup_GMDB) * t)
    return torch.maximum(A, base)

def hjb_residual(model, x, params, pinn_cfg):
    x.requires_grad_(True)

    use_cir = pinn_cfg["use_cir"]
    use_vi = pinn_cfg.get("use_vi", False) and params.has_GMWB

    t = x[:, 0:1]
    A = x[:, 1:2]
    W = x[:, 2:3]
    B_D = x[:, 3:4]
    v = x[:, 4:5]
    r = x[:, 5:6] if use_cir else torch.full_like(A, float(params.r))
    k = x[:, 6:7]

    V = model(x) * params.P

    grads = grad(V, x)
    Vt = grads[:, 0:1]
    VA = grads[:, 1:2]
    VW = grads[:, 2:3]
    VBD = grads[:, 3:4]
    Vv = grads[:, 4:5]
    Vk = grads[:, 6:7]
    VAA = grad(VA, x)[:, 1:2]
    Vvv = grad(Vv, x)[:, 4:5]
    VAv = grad(VA, x)[:, 4:5]
    Vkk = grad(Vk, x)[:, 6:7]

    if use_cir:
        Vr = grads[:, 5:6]
        Vrr = grad(Vr, x)[:, 5:6]
    else:
        Vr = None
        Vrr = None

    use_discrete = pinn_cfg.get("use_discrete_control", False) and params.has_GMWB
    if use_vi:
        gamma = torch.zeros_like(VA)
    elif use_discrete:
        gamma = None
    else:
        gamma = optimal_withdrawal(VA, VW, pinn_cfg["gamma_max"], pinn_cfg.get("gamma_k", 5.0))

    mu = hazard_rate_lc(t, k, params)

    v_pos = torch.clamp(v, min=0.0)
    sqrt_v = torch.sqrt(v_pos + 1e-12)
    A_safe = torch.clamp(A, min=1e-8)

    A_term = (r - params.phi - params.jump_compensator) * A * VA
    diffusion_A = 0.5 * v_pos * A**2 * VAA

    r_BD = float(getattr(params, "rollup_GMDB", 0.0))
    BD_term = r_BD * B_D * VBD if (params.has_GMWB and params.has_GMDB) else 0.0

    drift_v = params.kappa_v * (params.theta_v - v_pos) * Vv
    diffusion_v = 0.5 * params.sigma_v**2 * v_pos * Vvv
    cross_Av = params.rho * params.sigma_v * sqrt_v * A * VAv
    alpha_v = float(pinn_cfg.get("alpha_v", 1.0))
    drift_v = drift_v * alpha_v
    diffusion_v = diffusion_v * alpha_v
    cross_Av = cross_Av * alpha_v

    if use_cir:
        r_pos = torch.clamp(r, min=0.0)
        drift_r = params.kappa_r * (params.theta_r - r_pos) * Vr
        diffusion_r = 0.5 * params.sigma_r**2 * r_pos * Vrr
    else:
        drift_r = 0.0
        diffusion_r = 0.0

    mu_k = float(params.lc_model.drift)
    sigma_k = float(params.lc_model.volatility)
    if not pinn_cfg.get("use_stochastic_mortality", True):
        sigma_k = 0.0
    drift_k = mu_k * Vk
    diffusion_k = 0.5 * (sigma_k**2) * Vkk

    n_jump = pinn_cfg.get("n_jump", 8)
    if params.lam > 0 and n_jump > 0:
        y = torch.randn((A.shape[0], n_jump), device=A.device) * params.jump_std + params.jump_mean
        A_jump = A * torch.exp(y)
        t_jump = t.repeat(1, n_jump)
        W_jump = W.repeat(1, n_jump)
        BD_jump = B_D.repeat(1, n_jump)
        v_jump = v.repeat(1, n_jump)
        r_jump = r.repeat(1, n_jump) if use_cir else torch.full_like(A_jump, float(params.r))
        k_jump = k.repeat(1, n_jump)
        x_jump = torch.stack([t_jump, A_jump, W_jump, BD_jump, v_jump, r_jump, k_jump], dim=2).reshape(-1, 7)
        V_jump = model(x_jump).reshape(-1, n_jump).mean(dim=1, keepdim=True) * params.P
        jump_term = params.lam * (V_jump - V)
    else:
        jump_term = 0.0

    death = -mu * (death_benefit_contract(A, W, B_D, t, params) - V)

    base_hjb = (
        -Vt
        - A_term
        - BD_term
        - diffusion_A
        - drift_v
        - diffusion_v
        - cross_Av
        - drift_r
        - diffusion_r
        - drift_k
        - diffusion_k
        - jump_term
        + r * V
        + death
    )

    if use_discrete:
        n_gamma = int(pinn_cfg.get("n_gamma", 11))
        gamma_cap = torch.clamp(torch.minimum(torch.minimum(W, A_safe), torch.full_like(W, float(pinn_cfg["gamma_max"]))), min=0.0)
        frac = torch.linspace(0.0, 1.0, n_gamma, device=A.device).reshape(1, -1)
        gamma_grid = gamma_cap * frac
        extra_bd = (B_D / A_safe) * VBD if (params.has_GMWB and params.has_GMDB) else 0.0
        gamma_terms = gamma_grid * (VA + VW + extra_bd - 1.0)
        HJB = base_hjb + gamma_terms.max(dim=1, keepdim=True).values
    else:
        if use_vi:
            gamma_eff = torch.zeros_like(VA)
        else:
            gamma_eff = gamma
        extra_bd = (B_D / A_safe) * VBD if (params.has_GMWB and params.has_GMDB) else 0.0
        HJB = base_hjb + gamma_eff * (VA + VW + extra_bd - 1.0)

    if use_vi:
        D = VA + VW
        if params.has_GMWB and params.has_GMDB:
            D = D + (B_D / A_safe) * VBD
        G = D - 1.0
        return HJB, G

    return HJB

def loss_fn(model, x_int, x_T, params, pinn_cfg, x_W0=None, x_anni=None, w_T=5.0, w_W0=2.0, w_anni=10.0, return_components=False):
    res = hjb_residual(model, x_int, params, pinn_cfg)
    V_int = model(x_int) * params.P
    base_scale = torch.full_like(V_int, float(params.P))
    hjb_scale = base_scale

    if isinstance(res, tuple):
        F, G = res
        vi_weight = pinn_cfg.get("vi_weight", 5.0)
        F_s = F / hjb_scale
        G_s = G / hjb_scale
        min_FG = torch.minimum(F_s, G_s)
        loss_hjb = torch.mean(
            torch.relu(-F_s) ** 2
            + torch.relu(-G_s) ** 2
            + vi_weight * (min_FG ** 2)
        )
    else:
        res_s = res / hjb_scale
        loss_hjb = torch.mean(res_s**2)

    use_cir = pinn_cfg["use_cir"]
    VT = model(x_T) * params.P
    t_T = x_T[:, 0:1]
    A_T = x_T[:, 1:2]
    W_T = x_T[:, 2:3]
    B_D_T = x_T[:, 3:4]
    r_T = x_T[:, 5:6] if use_cir else torch.full_like(A_T, float(params.r))
    k_T = x_T[:, 6:7]
    ann_factors = pinn_cfg.get("annuity_factors")
    payoff = terminal_payoff_contract(
        A_T, W_T, B_D_T, t_T, r_T, k_T, params, use_cir=use_cir, ann_factors=ann_factors
    )
    _P = float(params.P)
    loss_T = torch.mean(((VT - payoff) / _P) ** 2)

    loss_W0 = 0.0
    if x_W0 is not None:
        VW0 = model(x_W0) * params.P
        A_W0 = x_W0[:, 1:2]
        loss_W0 = torch.mean(torch.relu((A_W0 - VW0) / _P) ** 2)

    loss_anni = 0.0
    if x_anni is not None:
        V_anni = model(x_anni) * params.P
        t_anni = x_anni[:, 0:1]
        A_anni = x_anni[:, 1:2]
        r_anni = x_anni[:, 5:6] if use_cir else torch.full_like(A_anni, float(params.r))
        k_anni = x_anni[:, 6:7]
        ann_val = annuity_value(
            A_anni, t_anni, r_anni, k_anni, params, use_cir=use_cir, ann_factors=ann_factors
        )
        loss_anni = torch.mean(torch.relu((ann_val - V_anni) / _P) ** 2)

    if return_components:
        return loss_hjb, loss_T, loss_W0, loss_anni
    return loss_hjb + w_T * loss_T + w_W0 * loss_W0 + w_anni * loss_anni

def _sample_base(n, max_base, active):
    if not active:
        return torch.zeros(n, 1, device=device)
    return torch.rand(n, 1, device=device) * max_base

def train_pinn(
    params,
    arch="valuenet",
    epochs=4000,
    n_int=4096,
    n_T=4000,
    n_W0=2000,
    n_anni=2000,
    n_annuity_paths=2000,
    use_stochastic_mortality=True,
    lr=1e-4,
    lr_step=2000,
    lr_gamma=0.6,
    grad_clip=5.0,
    pretrain_epochs=5000,
    w_T=300.0,
    w_W0=200.0,
    w_anni=200.0,
    w_pde=1.0,
    vi_weight=5.0,
    vi_focus_frac=0.3,
    gamma_k=2.0,
    n_gamma=21,
    alpha_v=1.0
):
    use_cir = params.use_stochastic_rates
    gamma_max = params.gmw_annual_frac * params.P if params.has_GMWB else 0.0
    W_max = params.gmw_total if params.has_GMWB else 0.0
    A_max = params.P * 3.0
    base_max = params.P * 3.0
    v_min = 0.0
    v_max = max(params.theta_v * 2.0, params.v0 * 2.0)
    T = float(params.T)

    if use_cir:
        r_min = 0.0
        r_max = max(params.theta_r + 4.0 * params.sigma_r, params.r * 2.0)
    else:
        r_min = params.r
        r_max = params.r

    k0 = float(params.lc_model.k_t[-1])
    mu_k = float(params.lc_model.drift)
    sigma_k = float(params.lc_model.volatility) if use_stochastic_mortality else 0.0
    k_min = k0 + mu_k * 0.0 - 4.0 * sigma_k * np.sqrt(T)
    k_max = k0 + mu_k * T + 4.0 * sigma_k * np.sqrt(T)

    annuity_factors = None
    if params.has_GMIB:
        q_paths_mc, _ = _build_q_paths_mc(params, n_annuity_paths, seed=24680)
        r_paths_mc = _build_r_paths_mc(params, n_annuity_paths, seed=13579)
        factors = []
        for t_step in range(params.T_steps + 1):
            factors.append(_annuity_factor_from_qpaths(q_paths_mc, params, path_start_year=t_step, r_paths=r_paths_mc))
        annuity_factors = torch.tensor(factors, device=device, dtype=torch.float32)

    pinn_cfg = {
        "use_cir": use_cir,
        "use_vi": False,
        "use_discrete_control": True,
        "n_gamma": int(n_gamma),
        "gamma_max": gamma_max,
        "gamma_k": gamma_k,
        "n_jump": 8,
        "use_stochastic_mortality": use_stochastic_mortality,
        "vi_weight": vi_weight,
        "vi_focus_frac": vi_focus_frac,
        "boundary_focus_frac": 0.0,
        "hjb_scale": params.P,
        "alpha_v": alpha_v,
        "annuity_factors": annuity_factors
    }

    x_min = torch.tensor([0.0, 0.0, 0.0, 0.0, v_min, r_min, k_min], device=device, dtype=torch.float32)
    x_max = torch.tensor([T, A_max, W_max, base_max, v_max, r_max, k_max], device=device, dtype=torch.float32)

    NetClass = DGMNet if arch == "dgm" else ValueNet
    model = NetClass(dim=7, x_min=x_min, x_max=x_max).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=lr_step, gamma=lr_gamma)

    def _sample_k(t_tensor):
        mu = k0 + mu_k * t_tensor
        if sigma_k <= 0.0:
            return mu
        std = sigma_k * torch.sqrt(torch.clamp(t_tensor, min=0.0))
        return mu + std * torch.randn_like(mu)

    def _det_base(t_tensor, rollup, active):
        if not active:
            return torch.zeros_like(t_tensor)
        return torch.full_like(t_tensor, float(params.P)) * torch.exp(float(rollup) * t_tensor)

    def _sample_interior_points(n_total):
        boundary_focus = pinn_cfg.get("boundary_focus_frac", pinn_cfg.get("vi_focus_frac", 0.0))
        n_focus = int(n_total * boundary_focus)
        n_uni = n_total - n_focus

        t_u = torch.rand(n_uni, 1, device=device) * T
        A_u = torch.rand(n_uni, 1, device=device) * A_max
        W_u = torch.rand(n_uni, 1, device=device) * W_max if params.has_GMWB else torch.zeros((n_uni, 1), device=device)
        if params.has_GMWB and params.has_GMDB:
            B_D_u = _sample_base(n_uni, base_max, True)
        else:
            B_D_u = torch.zeros((n_uni, 1), device=device)
        if alpha_v == 0.0:
            v_u = torch.full((n_uni, 1), float(params.v0), device=device)
        else:
            v_u = (torch.full((n_uni, 1), float(params.v0), device=device) if alpha_v == 0.0 else torch.rand(n_uni, 1, device=device) * (v_max - v_min) + v_min)
        if use_cir:
            r_u = torch.rand(n_uni, 1, device=device) * (r_max - r_min) + r_min
        else:
            r_u = torch.full((n_uni, 1), float(params.r), device=device)
        k_u = _sample_k(t_u)
        x_u = torch.cat([t_u, A_u, W_u, B_D_u, v_u, r_u, k_u], dim=1)

        if n_focus <= 0:
            return x_u

        t_f = torch.rand(n_focus, 1, device=device) * (0.2 * T)
        A_f = torch.normal(mean=float(params.P), std=float(0.25 * params.P), size=(n_focus, 1), device=device)
        A_f = torch.clamp(A_f, 0.0, A_max)
        W_f = torch.rand(n_focus, 1, device=device) * (0.1 * W_max)
        if params.has_GMWB and params.has_GMDB:
            B_D_f = _sample_base(n_focus, base_max, True)
        else:
            B_D_f = torch.zeros((n_focus, 1), device=device)
        if alpha_v == 0.0:
            v_f = torch.full((n_focus, 1), float(params.v0), device=device)
        else:
            v_f = (torch.full((n_focus, 1), float(params.v0), device=device) if alpha_v == 0.0 else torch.rand(n_focus, 1, device=device) * (v_max - v_min) + v_min)
        if use_cir:
            r_f = torch.rand(n_focus, 1, device=device) * (r_max - r_min) + r_min
        else:
            r_f = torch.full((n_focus, 1), float(params.r), device=device)
        k_f = _sample_k(t_f)
        x_f = torch.cat([t_f, A_f, W_f, B_D_f, v_f, r_f, k_f], dim=1)

        return torch.cat([x_u, x_f], dim=0)

    n_T_pre = max(n_T, 8000)
    n_W0_pre = max(n_W0, 4000)
    n_anni_pre = max(n_anni, 2000)
    for _ in tqdm(range(pretrain_epochs), desc="PINN pretrain", unit="epoch"):
        tT = torch.full((n_T_pre, 1), T, device=device)
        AT = torch.rand(n_T_pre, 1, device=device) * A_max
        WT = torch.rand(n_T_pre, 1, device=device) * W_max if params.has_GMWB else torch.zeros((n_T_pre, 1), device=device)
        if params.has_GMWB and params.has_GMDB:
            B_DT = _sample_base(n_T_pre, base_max, True)
        else:
            B_DT = torch.zeros((n_T_pre, 1), device=device)
        vT = (torch.full((n_T_pre, 1), float(params.v0), device=device) if alpha_v == 0.0 else torch.rand(n_T_pre, 1, device=device) * (v_max - v_min) + v_min)
        if use_cir:
            rT = torch.rand(n_T_pre, 1, device=device) * (r_max - r_min) + r_min
        else:
            rT = torch.full((n_T_pre, 1), float(params.r), device=device)
        kT = _sample_k(tT)
        x_T = torch.cat([tT, AT, WT, B_DT, vT, rT, kT], dim=1)

        x_W0 = None
        if params.has_GMWB:
            tW0 = torch.rand(n_W0_pre, 1, device=device) * T
            AW0 = torch.rand(n_W0_pre, 1, device=device) * A_max
            WW0 = torch.zeros(n_W0_pre, 1, device=device)
            if params.has_GMDB:
                BW_D0 = _sample_base(n_W0_pre, base_max, True)
            else:
                BW_D0 = torch.zeros((n_W0_pre, 1), device=device)
            vW0 = (torch.full((n_W0_pre, 1), float(params.v0), device=device) if alpha_v == 0.0 else torch.rand(n_W0_pre, 1, device=device) * (v_max - v_min) + v_min)
            if use_cir:
                rW0 = torch.rand(n_W0_pre, 1, device=device) * (r_max - r_min) + r_min
            else:
                rW0 = torch.full((n_W0_pre, 1), float(params.r), device=device)
            kW0 = _sample_k(tW0)
            x_W0 = torch.cat([tW0, AW0, WW0, BW_D0, vW0, rW0, kW0], dim=1)

        x_anni = None
        if params.has_GMIB:
            if params.dt < 1.0:
                steps_per_year = int(1 / params.dt)
                valid_steps = torch.arange(steps_per_year, params.T_steps, steps_per_year, device=device)
            else:
                valid_steps = torch.arange(1, params.T_steps, device=device)
            if valid_steps.numel() > 0:
                idx = torch.randint(0, valid_steps.numel(), (n_anni_pre,), device=device)
                steps = valid_steps[idx].float().unsqueeze(1)
                tA = steps * float(params.dt)
                AA = torch.rand(n_anni_pre, 1, device=device) * A_max
                WA = torch.zeros(n_anni_pre, 1, device=device)
                BD = torch.zeros((n_anni_pre, 1), device=device)
                vA = (torch.full((n_anni_pre, 1), float(params.v0), device=device) if alpha_v == 0.0 else torch.rand(n_anni_pre, 1, device=device) * (v_max - v_min) + v_min)
                if use_cir:
                    rA = torch.rand(n_anni_pre, 1, device=device) * (r_max - r_min) + r_min
                else:
                    rA = torch.full((n_anni_pre, 1), float(params.r), device=device)
                kA = _sample_k(tA)
                x_anni = torch.cat([tA, AA, WA, BD, vA, rA, kA], dim=1)

        use_cir_local = pinn_cfg["use_cir"]
        ann_factors = pinn_cfg.get("annuity_factors")
        V_T = model(x_T) * params.P
        t_T = x_T[:, 0:1]
        A_T = x_T[:, 1:2]
        W_T = x_T[:, 2:3]
        B_D_T = x_T[:, 3:4]
        r_T = x_T[:, 5:6] if use_cir_local else torch.full_like(A_T, float(params.r))
        k_T = x_T[:, 6:7]
        payoff_T = terminal_payoff_contract(
            A_T, W_T, B_D_T, t_T, r_T, k_T, params, use_cir=use_cir_local, ann_factors=ann_factors
        )
        _P = float(params.P)
        loss_T = torch.mean(((V_T - payoff_T) / _P) ** 2)

        loss_W0 = 0.0
        if x_W0 is not None:
            V_W0 = model(x_W0) * params.P
            A_W0 = x_W0[:, 1:2]
            loss_W0 = torch.mean(torch.relu((A_W0 - V_W0) / _P) ** 2)

        loss_anni = 0.0
        if x_anni is not None:
            V_anni = model(x_anni) * params.P
            t_anni = x_anni[:, 0:1]
            A_anni = x_anni[:, 1:2]
            r_anni = x_anni[:, 5:6] if use_cir_local else torch.full_like(A_anni, float(params.r))
            k_anni = x_anni[:, 6:7]
            ann_val = annuity_value(
                A_anni, t_anni, r_anni, k_anni, params, use_cir=use_cir_local, ann_factors=ann_factors
            )
            loss_anni = torch.mean(torch.relu((ann_val - V_anni) / _P) ** 2)

        loss = w_T * loss_T + w_W0 * loss_W0 + w_anni * loss_anni
        optimizer.zero_grad()
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

    for epoch in tqdm(range(epochs), desc="PINN training", unit="epoch"):
        x_int = _sample_interior_points(n_int)

        tT = torch.full((n_T, 1), T, device=device)
        AT = torch.rand(n_T, 1, device=device) * (params.P * 3.0)
        WT = torch.rand(n_T, 1, device=device) * W_max if params.has_GMWB else torch.zeros((n_T, 1), device=device)
        if params.has_GMWB and params.has_GMDB:
            B_DT = _sample_base(n_T, base_max, True)
        else:
            B_DT = torch.zeros((n_T, 1), device=device)
        vT = (torch.full((n_T, 1), float(params.v0), device=device) if alpha_v == 0.0 else torch.rand(n_T, 1, device=device) * (v_max - v_min) + v_min)
        if use_cir:
            rT = torch.rand(n_T, 1, device=device) * (r_max - r_min) + r_min
        else:
            rT = torch.full((n_T, 1), float(params.r), device=device)
        kT = _sample_k(tT)
        x_T = torch.cat([tT, AT, WT, B_DT, vT, rT, kT], dim=1)

        x_W0 = None
        if params.has_GMWB:
            tW0 = torch.rand(n_W0, 1, device=device) * T
            AW0 = torch.rand(n_W0, 1, device=device) * (params.P * 3.0)
            WW0 = torch.zeros(n_W0, 1, device=device)
            if params.has_GMDB:
                BW_D0 = _sample_base(n_W0, base_max, True)
            else:
                BW_D0 = torch.zeros((n_W0, 1), device=device)
            vW0 = (torch.full((n_W0, 1), float(params.v0), device=device) if alpha_v == 0.0 else torch.rand(n_W0, 1, device=device) * (v_max - v_min) + v_min)
            if use_cir:
                rW0 = torch.rand(n_W0, 1, device=device) * (r_max - r_min) + r_min
            else:
                rW0 = torch.full((n_W0, 1), float(params.r), device=device)
            kW0 = _sample_k(tW0)
            x_W0 = torch.cat([tW0, AW0, WW0, BW_D0, vW0, rW0, kW0], dim=1)

        x_anni = None
        if params.has_GMIB:
            if params.dt < 1.0:
                steps_per_year = int(1 / params.dt)
                valid_steps = torch.arange(steps_per_year, params.T_steps, steps_per_year, device=device)
            else:
                valid_steps = torch.arange(1, params.T_steps, device=device)
            if valid_steps.numel() > 0:
                idx = torch.randint(0, valid_steps.numel(), (n_anni,), device=device)
                steps = valid_steps[idx].float().unsqueeze(1)
                tA = steps * float(params.dt)
                AA = torch.rand(n_anni, 1, device=device) * A_max
                WA = torch.zeros(n_anni, 1, device=device)
                BD = torch.zeros((n_anni, 1), device=device)
                vA = (torch.full((n_anni, 1), float(params.v0), device=device) if alpha_v == 0.0 else torch.rand(n_anni, 1, device=device) * (v_max - v_min) + v_min)
                if use_cir:
                    rA = torch.rand(n_anni, 1, device=device) * (r_max - r_min) + r_min
                else:
                    rA = torch.full((n_anni, 1), float(params.r))
                kA = _sample_k(tA)
                x_anni = torch.cat([tA, AA, WA, BD, vA, rA, kA], dim=1)

        loss_hjb, loss_T, loss_W0, loss_anni = loss_fn(
            model, x_int, x_T, params, pinn_cfg, x_W0=x_W0, x_anni=x_anni,
            w_T=w_T, w_W0=w_W0, w_anni=w_anni, return_components=True
        )
        loss = w_pde * loss_hjb + w_T * loss_T + w_W0 * loss_W0 + w_anni * loss_anni

        optimizer.zero_grad()
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()
        if epoch % 200 == 0 or epoch == epochs - 1:
            tqdm.write(
                f"  ep {epoch:4d} | loss {loss.item():.4e} | "
                f"pde {loss_hjb.item():.4e} | term {loss_T.item():.4e} | "
                f"w0 {loss_W0.item() if isinstance(loss_W0, torch.Tensor) else 0.0:.4e}"
            )

    model.eval()
    with torch.no_grad():
        W0 = params.gmw_total if params.has_GMWB else 0.0
        BD0 = float(params.P) if (params.has_GMWB and params.has_GMDB) else 0.0
        x0 = torch.tensor([[0.0, params.P, W0, BD0, params.v0, params.r0, k0]], device=device, dtype=torch.float32)
        pinn_price = model(x0).item() * params.P
    return model, pinn_price

In [ ]:
# -------------------------
# 9) Common Infrastructure for Hedging
# -------------------------
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tensorflow as tf
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import norm as _norm
import time as _htime

BOND_MATURITY = 30.0 

# CIR ZCB
def _cir_B(tau, kappa, sigma):
    if tau <= 1e-8:
        return 0.0
    if sigma < 1e-10:
        return tau
    h = np.sqrt(kappa**2 + 2.0*sigma**2)
    exp_htau = np.exp(h * tau)
    return 2.0*(exp_htau - 1.0) / (2.0*h + (kappa + h)*(exp_htau - 1.0))

def cir_bond_price(r, tau, kappa, theta, sigma):
    r = np.asarray(r, dtype=np.float64)
    if tau <= 1e-8:
        return np.ones_like(r, dtype=np.float32)
    if sigma < 1e-10:
        return np.exp(-r * tau).astype(np.float32)
    h = np.sqrt(kappa**2 + 2.0*sigma**2)
    exp_htau = np.exp(h * tau)
    denom = 2.0*h + (kappa + h)*(exp_htau - 1.0)
    B     = 2.0*(exp_htau - 1.0) / denom
    logA  = (2.0*kappa*theta / sigma**2) * np.log(
                2.0*h * np.exp((kappa + h)*tau/2.0) / denom)
    return np.exp(logA - B*r).astype(np.float32)

def simulate_hedge_paths(params, han_policy, n_paths, seed):
    """
    Simulate market paths + policyholder behaviour for hedging.
    """
    rng = np.random.default_rng(seed + 31415)
    T, P, dt = params.T_steps, params.P, params.dt

    S, V_hes, q, R_rate, K_lc = simulate_paths(params, n_paths, seed=seed)

    gross_ret = (S[:, 1:] / (S[:, :-1] + 1e-12)).astype(np.float32)
    mm        = np.exp(R_rate[:, :T] * dt).astype(np.float32)
    r_integ   = np.cumsum(R_rate[:, :T] * dt, axis=1)
    disc      = np.exp(-r_integ).astype(np.float32)

    kap, tht, sig_r = params.kappa_r, params.theta_r, params.sigma_r
    bond_p    = np.stack([cir_bond_price(R_rate[:, t],   BOND_MATURITY - t*dt,       kap, tht, sig_r)
                          for t in range(T)], axis=1).astype(np.float32)
    bond_next = np.stack([cir_bond_price(R_rate[:, t+1], BOND_MATURITY - (t+1)*dt,   kap, tht, sig_r)
                          for t in range(T)], axis=1).astype(np.float32)
    bond_scale      = float(P)
    bond_excess_ret = ((bond_next - bond_p * mm) * bond_scale).astype(np.float32)
    bond_p          = (bond_p * bond_scale).astype(np.float32)

    A   = np.full(n_paths, float(P), np.float64)
    W   = np.full(n_paths, float(params.gmw_total), np.float64) if params.has_GMWB else np.zeros(n_paths)
    BD  = np.full(n_paths, float(P), np.float64)                if params.has_GMDB  else np.zeros(n_paths)
    alive = np.ones(n_paths, bool)

    A_rec  = np.zeros((n_paths, T), np.float32);  A_rec[:,  0] = A
    W_rec  = np.zeros((n_paths, T), np.float32);  W_rec[:,  0] = W
    BD_rec = np.zeros((n_paths, T), np.float32);  BD_rec[:, 0] = BD
    v_rec  = V_hes[:, :T].astype(np.float32)
    r_rec  = R_rate[:, :T].astype(np.float32)
    k_rec  = K_lc[:, :T].astype(np.float32)
    active      = np.ones((n_paths, T), np.float32)
    payout_disc = np.zeros(n_paths, np.float64)

    for t in range(T):
        alive_idx = np.where(alive)[0]
        if len(alive_idx) == 0:
            active[:, t:] = 0.0
            break
        disc_t = disc[:, t]

        # 1. GMWB withdrawal
        withdrawals = np.zeros(n_paths)
        if params.has_GMWB and han_policy is not None:
            t_yr  = np.full(len(alive_idx), (t + 1)*dt, np.float32)
            state = han_policy._create_state_vector(
                tf.constant(t_yr),
                tf.constant(A[alive_idx].astype(np.float32)),
                tf.constant(W[alive_idx].astype(np.float32)),
                tf.constant(BD[alive_idx].astype(np.float32)),
                tf.constant(R_rate[alive_idx, t+1].astype(np.float32)),
                tf.constant(V_hes[alive_idx,  t+1].astype(np.float32)),
                tf.constant(K_lc[alive_idx,   t+1].astype(np.float32)),
                params)
            w_raw = han_policy.models[t](state).numpy()[:, 0]
            cap   = params.gmw_step_frac * P
            w_raw = np.clip(w_raw, 0.0, np.minimum(cap, np.minimum(W[alive_idx], A[alive_idx])))
            withdrawals[alive_idx] = w_raw

        payout_disc[alive_idx] += withdrawals[alive_idx] * disc_t[alive_idx]
        A[alive_idx] -= withdrawals[alive_idx]
        if params.has_GMWB:
            W[alive_idx] -= withdrawals[alive_idx]
        if params.has_GMDB and params.has_GMWB:
            A_safe = np.maximum(A[alive_idx] + withdrawals[alive_idx], 1e-8)
            BD[alive_idx] = np.maximum(
                BD[alive_idx] * np.exp(params.rollup_GMDB*dt) * (1.0 - withdrawals[alive_idx]/A_safe), 0.0)

        # 2. GMIB annuitisation
        if params.has_GMIB and han_policy is not None and len(alive_idx) > 0:
            t_yr  = np.full(len(alive_idx), (t+1)*dt, np.float32)
            state = han_policy._create_state_vector(
                tf.constant(t_yr),
                tf.constant(A[alive_idx].astype(np.float32)),
                tf.constant(np.zeros(len(alive_idx), np.float32)),
                tf.constant(BD[alive_idx].astype(np.float32)),
                tf.constant(R_rate[alive_idx, t+1].astype(np.float32)),
                tf.constant(V_hes[alive_idx,  t+1].astype(np.float32)),
                tf.constant(K_lc[alive_idx,   t+1].astype(np.float32)),
                params)
            ann_prob = tf.nn.sigmoid(han_policy.models[t](state)).numpy()[:, 0]
            ann_idx  = alive_idx[ann_prob > 0.5]
            if len(ann_idx) > 0:
                age_t    = params.start_age + (t+1)*dt
                base_I_t = float(P)*np.exp(params.rollup_GMIB*(t+1)*dt)
                for pi in ann_idx:
                    pv_acc  = annuity_pv(A[pi]*params.ann_rate_current, 0, params.ann_max_age,
                                         params.r, age_t, dt, q[pi], t+1, R_rate[pi])
                    pv_guar = annuity_pv(base_I_t*params.ann_rate_guar, 0, params.ann_max_age,
                                         params.r, age_t, dt, q[pi], t+1, R_rate[pi])
                    payout_disc[pi] += max(A[pi], pv_acc, pv_guar) * disc_t[pi]
                alive[ann_idx]        = False
                active[ann_idx, t+1:] = 0.0
            alive_idx = np.where(alive)[0]

        # 3. Mortality
        if len(alive_idx) > 0:
            q_step   = 1.0 - (1.0 - q[alive_idx, t])**dt
            dies     = rng.random(len(alive_idx)) < q_step
            dies_idx = alive_idx[dies]
            if len(dies_idx) > 0:
                if params.has_GMWB and params.has_GMDB:
                    dp = np.maximum(BD[dies_idx], A[dies_idx]) + W[dies_idx]
                elif params.has_GMDB:
                    base_D = float(P)*np.exp(params.rollup_GMDB*(t+1)*dt)
                    dp     = np.maximum(base_D, A[dies_idx])
                else:
                    dp = A[dies_idx].copy()
                payout_disc[dies_idx] += dp * disc_t[dies_idx]
                alive[dies_idx]        = False
                active[dies_idx, t+1:] = 0.0

        # 4. Market update
        surv_idx = np.where(alive)[0]
        if len(surv_idx) > 0:
            A[surv_idx] = np.maximum(A[surv_idx]*gross_ret[surv_idx, t]*(1.0 - params.phi*dt), 0.0)

        if t + 1 < T:
            A_rec[:, t+1]  = A
            W_rec[:, t+1]  = W
            BD_rec[:, t+1] = BD

    # Terminal payoff
    alive_idx = np.where(alive)[0]
    if len(alive_idx) > 0:
        fp = A[alive_idx].copy()
        if params.has_GMWB:
            fp += W[alive_idx]
        if params.has_GMAB:
            fp = np.maximum(fp, float(P)*np.exp(params.rollup_GMAB*params.T))
        if params.has_GMIB:
            age_T    = params.start_age + params.T
            base_I_T = float(P)*np.exp(params.rollup_GMIB*params.T)
            for i, pi in enumerate(alive_idx):
                pv_acc  = annuity_pv(A[pi]*params.ann_rate_current, 0, params.ann_max_age,
                                     params.r, age_T, dt, q[pi], params.T_steps, R_rate[pi])
                pv_guar = annuity_pv(base_I_T*params.ann_rate_guar, 0, params.ann_max_age,
                                     params.r, age_T, dt, q[pi], params.T_steps, R_rate[pi])
                fp[i] = max(fp[i], pv_acc, pv_guar)
        payout_disc[alive_idx] += fp * disc[:, -1][alive_idx]

    return {
        'gross_ret':       gross_ret,
        'mm':              mm,
        'disc':            disc,
        'A':               A_rec,
        'W':               W_rec,
        'BD':              BD_rec,
        'v':               v_rec,
        'r':               r_rec,
        'k':               k_rec,
        'active':          active,
        'payout_disc':     payout_disc.astype(np.float32),
        'bond_p':          bond_p,
        'bond_excess_ret': bond_excess_ret,
    }


# ── P&L ───────────────────────────────────────────────────────────────────────

def compute_pnl(delta_arr, data, p0, params, tc=0.0,
                delta_r_arr=None, delta_C_arr=None):
    """
    Discounted P&L across all hedging instruments.
    """
    n, T   = delta_arr.shape
    pnl    = np.full(n, float(p0), np.float64)
    prev_S = np.zeros(n, np.float64)
    prev_r = np.zeros(n, np.float64)

    for t in range(T):
        d_S  = delta_arr[:, t].astype(np.float64)
        A_t  = data['A'][:, t].astype(np.float64)
        R    = data['gross_ret'][:, t].astype(np.float64)
        mm_t = data['mm'][:, t].astype(np.float64)
        dsc  = data['disc'][:, t].astype(np.float64)
        act  = data['active'][:, t].astype(np.float64)

        gain = d_S * A_t * (R - mm_t) * dsc * act
        if tc > 0.0:
            gain -= tc * np.abs(d_S - prev_S) * A_t * dsc * act

        if delta_r_arr is not None:
            d_r  = delta_r_arr[:, t].astype(np.float64)
            ber  = data['bond_excess_ret'][:, t].astype(np.float64)
            bp   = data['bond_p'][:, t].astype(np.float64)
            gain += d_r * ber * dsc * act
            if tc > 0.0:
                gain -= tc * np.abs(d_r - prev_r) * bp * dsc * act
            prev_r = d_r

        pnl   += gain
        prev_S = d_S

    pnl -= data['payout_disc'].astype(np.float64)
    return pnl


def cvar_stats(pnl, alpha=0.95):
    q_low = np.percentile(pnl, (1.0 - alpha)*100.0)
    cvar  = -float(pnl[pnl <= q_low].mean())
    return {'mean_pnl': float(pnl.mean()), 'std_pnl': float(pnl.std()), 'cvar95': cvar}

In [ ]:
# -------------------------
# 10) DP Method - HEDGING
# -------------------------

def _make_interp(grids, V_t):
    keep  = [i for i, g in enumerate(grids) if len(g) > 1]
    V_sq  = V_t.copy()
    for ax in reversed([i for i in range(len(grids)) if i not in keep]):
        V_sq = np.squeeze(V_sq, axis=ax)
    interp = RegularGridInterpolator(
        tuple(grids[i] for i in keep), V_sq,
        method='linear', bounds_error=False, fill_value=None)
    return interp, keep

def dp_delta_hedge(dp_result, data, p0, params, tc=0.0, eps_frac=0.05):
    A_grid = dp_result['A_grid']
    V_arr  = dp_result['V']

    has_G = bool(getattr(params, 'has_GMWB', False)) and ('G_grid' in dp_result)
    has_B = has_G and ('B_grid' in dp_result)

    n, T  = data['A'].shape
    P_nom = float(params.P)
    eps_A = eps_frac * P_nom

    delta_arr   = np.zeros((n, T), np.float64)
    delta_r_arr = np.zeros((n, T), np.float64)

    for t in range(T):
        tau_t = max(BOND_MATURITY - t * params.dt, 1e-8)
        B_val = _cir_B(tau_t, params.kappa_r, params.sigma_r)
        A_t  = data['A'][:, t].astype(np.float64)
        v_t  = data['v'][:, t]
        r_t  = data['r'][:, t]
        k_t  = data['k'][:, t]
        act  = data['active'][:, t].astype(np.float64)
        bp_t = data['bond_p'][:, t].astype(np.float64)

        A_p = np.clip(A_t + eps_A, A_grid[0], A_grid[-1])
        A_m = np.clip(A_t - eps_A, A_grid[0], A_grid[-1])

        if has_G and has_B:
            G_grid = dp_result['G_grid'];  B_grid = dp_result['B_grid']
            V_grid = dp_result['V_grid'];  R_grid = dp_result['R_grid'];  K_grid = dp_result['K_grid']
            W_t  = data['W'][:, t];  BD_t = data['BD'][:, t]
            G_c  = np.clip(W_t,  G_grid[0], G_grid[-1])
            B_c  = np.clip(BD_t, B_grid[0], B_grid[-1])
            v_c  = np.clip(v_t,  V_grid[0], V_grid[-1])
            r_c  = np.clip(r_t,  R_grid[0], R_grid[-1])
            k_c  = np.clip(k_t,  K_grid[0], K_grid[-1])
            all_grids = [A_grid, G_grid, B_grid, V_grid, R_grid, K_grid]
            base_c    = [A_t,    G_c,    B_c,    v_c,    r_c,    k_c]
            r_idx     = 4
        elif has_G:
            G_grid = dp_result['G_grid']
            V_grid = dp_result['V_grid'];  R_grid = dp_result['R_grid'];  K_grid = dp_result['K_grid']
            W_t  = data['W'][:, t]
            G_c  = np.clip(W_t, G_grid[0], G_grid[-1])
            v_c  = np.clip(v_t, V_grid[0], V_grid[-1])
            r_c  = np.clip(r_t, R_grid[0], R_grid[-1])
            k_c  = np.clip(k_t, K_grid[0], K_grid[-1])
            all_grids = [A_grid, G_grid, V_grid, R_grid, K_grid]
            base_c    = [A_t,    G_c,    v_c,    r_c,    k_c]
            r_idx     = 3
        else:
            V_grid = dp_result['V_grid'];  R_grid = dp_result['R_grid'];  K_grid = dp_result['K_grid']
            v_c  = np.clip(v_t, V_grid[0], V_grid[-1])
            r_c  = np.clip(r_t, R_grid[0], R_grid[-1])
            k_c  = np.clip(k_t, K_grid[0], K_grid[-1])
            all_grids = [A_grid, V_grid, R_grid, K_grid]
            base_c    = [A_t,    v_c,    r_c,    k_c]
            r_idx     = 2

        interp, keep = _make_interp(all_grids, V_arr[t])

        base_pA = base_c.copy(); base_pA[0] = A_p
        base_mA = base_c.copy(); base_mA[0] = A_m
        pts_pA  = np.stack([base_pA[i] for i in keep], axis=1)
        pts_mA  = np.stack([base_mA[i] for i in keep], axis=1)
        V_pA    = interp(pts_pA).astype(np.float64)
        V_mA    = interp(pts_mA).astype(np.float64)
        delta_arr[:, t] = (V_pA - V_mA) / np.maximum(A_p - A_m, 1.0) * act

        if r_idx in keep:
            eps_r   = eps_frac * (R_grid[-1] - R_grid[0])
            r_p     = np.clip(r_t + eps_r, R_grid[0], R_grid[-1])
            r_m     = np.clip(r_t - eps_r, R_grid[0], R_grid[-1])
            base_pr = base_c.copy(); base_pr[r_idx] = r_p
            base_mr = base_c.copy(); base_mr[r_idx] = r_m
            pts_pr  = np.stack([base_pr[i] for i in keep], axis=1)
            pts_mr  = np.stack([base_mr[i] for i in keep], axis=1)
            V_pr    = interp(pts_pr).astype(np.float64)
            V_mr    = interp(pts_mr).astype(np.float64)
            dV_dr   = (V_pr - V_mr) / np.maximum(r_p - r_m, 1e-10)
            delta_r_arr[:, t] = dV_dr / (-B_val * bp_t) * act

    pnl = compute_pnl(delta_arr, data, p0, params, tc=tc, delta_r_arr=delta_r_arr)
    return {'delta': delta_arr, 'delta_r': delta_r_arr, 'pnl': pnl, **cvar_stats(pnl)}

In [ ]:
# -------------------------
# 11) PINN Method - HEDGING
# -------------------------

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

def _h_abs(x, eps=1e-3):
    """Smooth |x|."""
    return torch.sqrt(x ** 2 + eps ** 2) - eps

def _h_grad(x, eps=1e-3):
    """Derivative of _h_abs."""
    return x / torch.sqrt(x ** 2 + eps ** 2)

def _h_grad2(x, eps=1e-3):
    """Second derivative of _h_abs."""
    return eps ** 2 / (x ** 2 + eps ** 2) ** 1.5

def _newton_1d(g, mu_hat, eps_huber, n_iter=6):
    x = g.clone()
    for _ in range(n_iter):
        Fx  = x - mu_hat * _h_grad(x, eps_huber) - g
        Fpx = 1.0 - mu_hat * _h_grad2(x, eps_huber)
        Fpx = torch.clamp(Fpx, min=1e-6)
        x   = x - Fx / Fpx
    return x

def _eval_Q(dS, dr, d2cc_p, d2Ac_p, d2rc_p,
            A_p, r_p, sr, Bp,
            prev_dS, prev_dr,
            use_bond, lam, P_nom, eps_huber):
    """Evaluate the (normalised) Hamiltonian Q at given positions."""
    Q = 0.5 * d2cc_p * dS ** 2 + dS * d2Ac_p
    if lam > 1e-12:
        Q = Q 

    if use_bond:
        u_r = -dr * Bp * sr * torch.sqrt(torch.clamp(r_p, min=1e-10))
        Q = Q + 0.5 * d2cc_p * u_r ** 2 + u_r * sr * torch.sqrt(
            torch.clamp(r_p, min=1e-10)) * d2rc_p
    return Q

class HedgingHJBNet(nn.Module):

    def __init__(self, n_in=8, n_hidden=128, n_layers=4):
        super().__init__()
        self.fc_in  = nn.Linear(n_in, n_hidden)
        self.hidden = nn.ModuleList(
            [nn.Linear(n_hidden, n_hidden) for _ in range(n_layers - 1)])
        self.skip   = nn.Linear(n_in, n_hidden)
        self.fc_out = nn.Linear(n_hidden, 1)
        nn.init.zeros_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, x):
        h = torch.tanh(self.fc_in(x))
        for layer in self.hidden:
            h = torch.tanh(layer(h))
        h = h + torch.tanh(self.skip(x))
        return self.fc_out(h).squeeze(-1)

def train_hjb_pinn_hedge(data, p0, params, alpha=0.95,
                         instruments='SB',
                         tc=0.0,
                         eval_tc=None,
                         eps_huber=1e-3,
                         n_epochs=1000, pretrain_epochs=100,
                         lr=5e-4, lr_step=250, lr_gamma=0.5,
                         n_pde=4096, n_term=4096,
                         n_hidden=128, n_layers=4,
                         w_pde=10.0, w_term=50.0,
                         device='cpu', seed=0, verbose=True,
                         init_model=None,
                         prev_delta_S_paths=None,
                         prev_delta_r_paths=None):

    torch.manual_seed(seed)
    np.random.seed(seed)

    use_bond = 'B' in instruments
    use_call = 'C' in instruments
    use_tc   = tc > 1e-10

    n_prev = 0
    if use_tc:
        n_prev = 1 + (1 if use_bond else 0) 
    n_in = 8 + n_prev

    T_steps = params.T_steps
    P_nom   = float(params.P)
    T_phys  = float(params.T)
    phi_p   = float(params.phi)
    v0      = max(float(getattr(params, 'theta_v', 0.0225)), 1e-6)
    r0      = max(float(params.theta_r), 1e-6)
    k0      = float(params.lc_model.k_t[-1])
    sk_val  = (max(float(params.lc_model.volatility) * float(params.T) ** 0.5, 1e-6)
               if params.use_stochastic_mortality else 1.0)

    kv = float(params.kappa_v);  thv = float(params.theta_v);  sv = float(params.sigma_v)
    kr = float(params.kappa_r);  thr = float(params.theta_r);  sr = float(params.sigma_r)
    rho_heston = float(getattr(params, 'rho', -0.7))
    mu_k  = float(getattr(params.lc_model, 'mu_k', 0.0))
    sig_k = float(params.lc_model.volatility) if params.use_stochastic_mortality else 0.0

    B_tau   = float(_cir_B(BOND_MATURITY, kr, sr))
    n_paths = data['A'].shape[0]

    s_A = 1.0 / P_nom;  s_v = 1.0 / v0;  s_r = 1.0 / r0
    s_k = 1.0 / sk_val; s_t = 1.0 / T_phys

    DELTA_CLIP_S = 10.0   # position clip for stock delta
    DELTA_CLIP_R =  3.0   # position clip for bond delta
    PREV_SCALE   = 10.0   # normalisation for prev-position inputs

    def _base_feat(t_n, idx, t_idx, c_phys):
        """8 base features (no prev positions)."""
        return np.stack([
            t_n,
            data['A'][idx, t_idx] / P_nom,
            data['W'][idx, t_idx] / P_nom,
            data['BD'][idx, t_idx] / P_nom,
            np.clip(data['v'][idx, t_idx] / v0,              0.0,  5.0),
            np.clip(data['r'][idx, t_idx] / r0,              0.0,  5.0),
            np.clip((data['k'][idx, t_idx] - k0) / sk_val,  -3.0,  3.0),
            np.clip(c_phys / P_nom,                          -3.0,  4.0),
        ], axis=1).astype(np.float32)

    def _sample_prev(idx, t_idx, paths, clip, scale, n):
        """Sample prev_delta from path trajectories when available."""
        if paths is not None:
            prev = np.where(t_idx > 0,
                            paths[idx, np.maximum(t_idx - 1, 0)],
                            0.0).astype(np.float32)
            prev = prev + np.random.normal(0.0, clip * 0.05, n).astype(np.float32)
            prev = np.clip(prev, -clip, clip)
        else:
            prev = np.random.uniform(-clip, clip, n).astype(np.float32)
        return (prev / scale).reshape(-1, 1)

    def sample_interior(n):
        idx   = np.random.randint(0, n_paths, n)
        t_idx = np.random.randint(0, T_steps - 1, n)
        feat  = _base_feat(t_idx / T_steps, idx, t_idx,
                           np.random.uniform(-1.5 * P_nom, 2.5 * P_nom, n))
        if use_tc:
            cols = [feat,
                    _sample_prev(idx, t_idx, prev_delta_S_paths,
                                 DELTA_CLIP_S, PREV_SCALE, n)]
            if use_bond:
                cols.append(_sample_prev(idx, t_idx, prev_delta_r_paths,
                                         DELTA_CLIP_R, PREV_SCALE, n))
            feat = np.concatenate(cols, axis=1)
        return torch.tensor(feat, dtype=torch.float32, device=device)

    def sample_terminal(n):
        idx    = np.random.randint(0, n_paths, n)
        t_idx  = np.full(n, T_steps - 1, dtype=int)
        c_phys = np.random.uniform(-1.5 * P_nom, 2.5 * P_nom, n)
        feat   = _base_feat(np.ones(n), idx, t_idx, c_phys)
        if use_tc:
            cols = [feat,
                    _sample_prev(idx, t_idx, prev_delta_S_paths,
                                 DELTA_CLIP_S, PREV_SCALE, n)]
            if use_bond:
                cols.append(_sample_prev(idx, t_idx, prev_delta_r_paths,
                                         DELTA_CLIP_R, PREV_SCALE, n))
            feat = np.concatenate(cols, axis=1)
        L_i    = data['payout_disc'][idx].astype(np.float64)
        arg    = (-float(p0) - c_phys + L_i) / P_nom
        beta   = 20.0
        target = np.log1p(np.exp(np.clip(arg * beta, -50., 50.))) / beta / (1.0 - alpha)
        feat_t = torch.tensor(feat, dtype=torch.float32, device=device)
        return feat_t, torch.tensor(target.astype(np.float32), device=device)

    model = HedgingHJBNet(n_in=n_in, n_hidden=n_hidden, n_layers=n_layers).to(device)
    if init_model is not None:
        src_sd = init_model.state_dict()
        dst_sd = model.state_dict()
        for key in dst_sd:
            if key not in src_sd:
                continue
            if src_sd[key].shape == dst_sd[key].shape:
                dst_sd[key].copy_(src_sd[key])          
            elif src_sd[key].dim() == 2:                
                n_src = src_sd[key].shape[1]
                dst_sd[key][:, :n_src].copy_(src_sd[key])   
                dst_sd[key][:, n_src:].zero_()    
        model.load_state_dict(dst_sd)

    opt   = optim.Adam(model.parameters(), lr=lr)

    if verbose and pretrain_epochs > 0:
        print('    pretrain {}...'.format(pretrain_epochs), end=' ', flush=True)
    loss = torch.tensor(0.0)
    for _ in range(pretrain_epochs):
        model.train()
        x_T, y_T = sample_terminal(n_term * 2)
        loss = torch.mean((model(x_T) - y_T) ** 2)
        opt.zero_grad(); loss.backward(); opt.step()
    if verbose and pretrain_epochs > 0:
        print('done  term_loss={:.3e}'.format(loss.item()))

    sched = optim.lr_scheduler.StepLR(opt, step_size=lr_step, gamma=lr_gamma)

    for epoch in range(1, n_epochs + 1):
        model.train()

        x_T, y_T = sample_terminal(n_term)
        loss_term = torch.mean((model(x_T) - y_T) ** 2)

        x_int = sample_interior(n_pde)
        x_int.requires_grad_(True)

        A_p = x_int[:, 1] * P_nom
        v_p = x_int[:, 4] * v0
        r_p = x_int[:, 5] * r0

        H_val = model(x_int)
        g1    = torch.autograd.grad(H_val.sum(), x_int, create_graph=True)[0]

        g_dc  = torch.autograd.grad(g1[:, 7].sum(), x_int, create_graph=True)[0]
        d2_cc = g_dc[:, 7]
        d2_Ac = g_dc[:, 1]
        d2_vc = g_dc[:, 4]
        d2_rc = g_dc[:, 5]

        g_dA  = torch.autograd.grad(g1[:, 1].sum(), x_int, create_graph=True)[0]
        d2_AA = g_dA[:, 1]; d2_Av = g_dA[:, 4]
        g_dv  = torch.autograd.grad(g1[:, 4].sum(), x_int, create_graph=True)[0]
        d2_vv = g_dv[:, 4]
        g_dr  = torch.autograd.grad(g1[:, 5].sum(), x_int, create_graph=True)[0]
        d2_rr = g_dr[:, 5]
        d2_Ar = g_dr[:, 1] 

        Lx_H  = (r_p - phi_p) * A_p * (g1[:, 1] * s_A)
        Lx_H += 0.5 * A_p ** 2 * v_p   * (d2_AA * s_A ** 2)
        Lx_H += kv * (thv - v_p)        * (g1[:, 4] * s_v)
        Lx_H += 0.5 * sv ** 2 * v_p     * (d2_vv * s_v ** 2)
        Lx_H += kr * (thr - r_p)        * (g1[:, 5] * s_r)
        Lx_H += 0.5 * sr ** 2 * r_p     * (d2_rr * s_r ** 2)
        Lx_H += rho_heston * A_p * sv * v_p * (d2_Av * s_A * s_v)
        rho_Sr_val = float(getattr(params, 'rho_Sr', 0.0))
        if abs(rho_Sr_val) > 1e-10:
            Lx_H += rho_Sr_val * A_p * torch.sqrt(torch.clamp(v_p, min=1e-10)) \
                    * sr * torch.sqrt(torch.clamp(r_p, min=1e-10)) \
                    * (d2_Ar * s_A * s_r)
        if params.use_stochastic_mortality and sig_k > 1e-10:
            g_dk  = torch.autograd.grad(g1[:, 6].sum(), x_int, create_graph=True)[0]
            Lx_H += mu_k * (g1[:, 6] * s_k)
            Lx_H += 0.5 * sig_k ** 2 * (g_dk[:, 6] * s_k ** 2)

        eps_cc    = 1e-8
        d2cc_phys = d2_cc * s_A ** 2
        d2cc_safe = torch.clamp(d2cc_phys, min=eps_cc)
        d2Ac_p    = d2_Ac * s_A ** 2
        d2rc_p    = d2_rc * s_A * s_r
        d2vc_p    = d2_vc * s_A * s_v
        Hc        = g1[:, 7] 
        if not use_tc:
            ham = (d2Ac_p) ** 2 / (2.0 * d2cc_safe)
            if use_bond:
                ham += sr ** 2 * r_p * d2rc_p ** 2 / (2.0 * d2cc_safe)
            if use_call:
                ham += sv ** 2 * v_p * d2vc_p ** 2 / (2.0 * d2cc_safe)

        else:
            prev_dS = x_int[:, 8] * PREV_SCALE
            prev_dr = x_int[:, 9] * PREV_SCALE if use_bond else torch.zeros_like(A_p)

            a_S = -d2Ac_p / d2cc_safe

            mu_S = tc * A_p * Hc / (P_nom * d2cc_safe)
            mu_S = torch.clamp(mu_S, min=-50.0, max=0.0)

            g_S = a_S - prev_dS
            x_S = _newton_1d(g_S, mu_S, eps_huber)
            dS_opt = torch.clamp(prev_dS + x_S, -DELTA_CLIP_S, DELTA_CLIP_S)

            Q_A = 0.5 * d2cc_safe * dS_opt ** 2 + dS_opt * d2Ac_p
            Q_A = Q_A - tc * A_p * Hc / P_nom * _h_abs(dS_opt - prev_dS, eps_huber)

            ham = -Q_A 

            if use_bond:
                sr_r = sr * torch.sqrt(torch.clamp(r_p, min=1e-10))
                a_r_u = d2rc_p / (d2cc_safe * sr_r)
                mu_r = tc * Hc / (P_nom * d2cc_safe * sr_r)
                mu_r = torch.clamp(mu_r, min=-50.0, max=0.0)

                g_r  = a_r_u - prev_dr
                x_r  = _newton_1d(g_r, mu_r, eps_huber)
                dr_opt = torch.clamp(prev_dr + x_r, -DELTA_CLIP_R, DELTA_CLIP_R)

                u_r   = -dr_opt * sr_r
                Q_r   = 0.5 * d2cc_safe * u_r ** 2 + u_r * sr_r * d2rc_p
                Q_r   = Q_r - tc * 1.0 * Hc / P_nom * _h_abs(dr_opt - prev_dr, eps_huber)
                ham   = ham + (-Q_r)

        dH_dt_phys = g1[:, 0] * s_t
        residual   = dH_dt_phys + Lx_H - ham
        loss_pde   = torch.mean((residual / P_nom) ** 2)

        loss = w_term * loss_term + w_pde * loss_pde
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); sched.step()

        if verbose and epoch % 100 == 0:
            print('    epoch {:4d}  term={:.3e}  pde={:.3e}'.format(
                epoch, loss_term.item(), loss_pde.item()))

    _eval_tc = eval_tc if eval_tc is not None else tc

    model.eval()
    dS_out = np.zeros((n_paths, T_steps), np.float32)
    dr_out = np.zeros((n_paths, T_steps), np.float32)
    dC_out = np.zeros((n_paths, T_steps), np.float32)
    pnl_running = np.full(n_paths, float(p0), np.float32)

    prev_dS_ev = np.zeros(n_paths, np.float32)
    prev_dr_ev = np.zeros(n_paths, np.float32)
    prev_dC_ev = np.zeros(n_paths, np.float32)

    for t in range(T_steps):
        base_np = np.stack([
            np.full(n_paths, t / T_steps),
            data['A'][:, t] / P_nom,
            data['W'][:, t] / P_nom,
            data['BD'][:, t] / P_nom,
            np.clip(data['v'][:, t] / v0,              0.0,  5.0),
            np.clip(data['r'][:, t] / r0,              0.0,  5.0),
            np.clip((data['k'][:, t] - k0) / sk_val,  -3.0,  3.0),
            np.clip(pnl_running / P_nom,               -3.0,  4.0),
        ], axis=1).astype(np.float32)

        if use_tc:
            cols = [base_np,
                    np.clip(prev_dS_ev / PREV_SCALE, -1.0, 1.0).reshape(-1, 1)]
            if use_bond:
                cols.append(np.clip(prev_dr_ev / PREV_SCALE, -1.0, 1.0).reshape(-1, 1))
            feat_np = np.concatenate(cols, axis=1)
        else:
            feat_np = base_np

        x_t  = torch.tensor(feat_np, dtype=torch.float32, device=device,
                             requires_grad=True)
        H_t  = model(x_t)
        g1_t = torch.autograd.grad(H_t.sum(), x_t, create_graph=True)[0]
        g_dc_t = torch.autograd.grad(g1_t[:, 7].sum(), x_t)[0]

        d2cc_n = g_dc_t[:, 7].detach().cpu().numpy()
        d2Ac_n = g_dc_t[:, 1].detach().cpu().numpy()
        d2vc_n = g_dc_t[:, 4].detach().cpu().numpy()
        d2rc_n = g_dc_t[:, 5].detach().cpu().numpy()
        Hc_n   = g1_t[:, 7].detach().cpu().numpy()

        d2cc_raw = d2cc_n * s_A ** 2
        d2cc_p = np.where(np.abs(d2cc_raw) > 1e-10,
                          d2cc_raw,
                          np.sign(d2cc_raw + 1e-20) * 1e-10)
        d2Ac_p = d2Ac_n * s_A ** 2
        d2vc_p = d2vc_n * s_A * s_v
        d2rc_p = d2rc_n * s_A * s_r

        act_t  = data['active'][:, t]
        A_t    = data['A'][:, t].astype(np.float64)
        r_t    = data['r'][:, t].astype(np.float64)

        if not use_tc:
            if _eval_tc > 1e-12:
                a_S   = np.clip(-d2Ac_p / d2cc_p, -DELTA_CLIP_S, DELTA_CLIP_S)
                mu_S  = np.clip(_eval_tc * A_t * Hc_n / (P_nom * d2cc_p), -50.0, 0.0)
                g_S   = a_S - prev_dS_ev
                x_S   = g_S.copy()
                for _ in range(6):
                    h_g  = x_S / np.sqrt(x_S ** 2 + eps_huber ** 2)
                    h_g2 = eps_huber ** 2 / (x_S ** 2 + eps_huber ** 2) ** 1.5
                    F    = x_S - mu_S * h_g - g_S
                    Fp   = np.clip(1.0 - mu_S * h_g2, 1e-6, None)
                    x_S  = x_S - F / Fp
                delta_S_t = np.clip(prev_dS_ev + x_S, -DELTA_CLIP_S, DELTA_CLIP_S) * act_t

                if use_bond:
                    bp_t  = data['bond_p'][:, t].astype(np.float64)
                    Bp    = B_tau * bp_t
                    Bp_s  = np.where(np.abs(Bp) > 1e-10, Bp, 1e-10)
                    a_r   = np.clip(d2rc_p / (Bp_s * d2cc_p), -DELTA_CLIP_R, DELTA_CLIP_R)
                    mu_r  = np.clip(_eval_tc * Hc_n / (P_nom * Bp_s * d2cc_p), -50.0, 0.0)
                    g_r   = a_r - prev_dr_ev
                    x_r   = g_r.copy()
                    for _ in range(6):
                        h_g  = x_r / np.sqrt(x_r ** 2 + eps_huber ** 2)
                        h_g2 = eps_huber ** 2 / (x_r ** 2 + eps_huber ** 2) ** 1.5
                        F    = x_r - mu_r * h_g - g_r
                        Fp   = np.clip(1.0 - mu_r * h_g2, 1e-6, None)
                        x_r  = x_r - F / Fp
                    delta_r_t = np.clip(prev_dr_ev + x_r, -DELTA_CLIP_R, DELTA_CLIP_R) * act_t
                else:
                    delta_r_t = np.zeros(n_paths)
            else:
                delta_S_t = -d2Ac_p / d2cc_p
                delta_S_t = np.clip(delta_S_t, -DELTA_CLIP_S, DELTA_CLIP_S) * act_t

                if use_bond:
                    bp_t  = data['bond_p'][:, t].astype(np.float64)
                    Bp    = B_tau * bp_t
                    Bp_s  = np.where(np.abs(Bp) > 1e-10, Bp, 1e-10)
                    if use_call:
                        delta_r_t = (d2rc_p - d2vc_p * Rc / np.where(np.abs(Vc) > 1e-8, Vc, 1e-8)) / (Bp_s * d2cc_p)
                    else:
                        delta_r_t = d2rc_p / (Bp_s * d2cc_p)
                    delta_r_t = np.clip(delta_r_t, -DELTA_CLIP_R, DELTA_CLIP_R) * act_t
                else:
                    delta_r_t = np.zeros(n_paths)

        else:
            a_S   = -d2Ac_p / d2cc_p
            mu_S  = np.clip(tc * A_t * Hc_n / (P_nom * d2cc_p), -50.0, 0.0)
            g_S   = a_S - prev_dS_ev

            x_S = g_S.copy()
            for _ in range(6):
                h_g   = x_S / np.sqrt(x_S ** 2 + eps_huber ** 2)
                h_g2  = eps_huber ** 2 / (x_S ** 2 + eps_huber ** 2) ** 1.5
                F     = x_S - mu_S * h_g - g_S
                Fp    = np.clip(1.0 - mu_S * h_g2, 1e-6, None)
                x_S   = x_S - F / Fp
            delta_S_t = np.clip(prev_dS_ev + x_S, -DELTA_CLIP_S, DELTA_CLIP_S) * act_t

            if use_bond:
                bp_t = data['bond_p'][:, t].astype(np.float64)
                Bp   = B_tau * bp_t
                sr_r = sr * np.sqrt(np.maximum(r_t, 1e-10))
                Bp_s = np.where(np.abs(Bp) > 1e-10, Bp, 1e-10)
                a_r  = d2rc_p / (d2cc_p * sr_r)
                mu_r = np.clip(tc * Hc_n / (P_nom * d2cc_p * sr_r), -50.0, 0.0)
                g_r  = a_r - prev_dr_ev

                x_r = g_r.copy()
                for _ in range(6):
                    h_g   = x_r / np.sqrt(x_r ** 2 + eps_huber ** 2)
                    h_g2  = eps_huber ** 2 / (x_r ** 2 + eps_huber ** 2) ** 1.5
                    F     = x_r - mu_r * h_g - g_r
                    Fp    = np.clip(1.0 - mu_r * h_g2, 1e-6, None)
                    x_r   = x_r - F / Fp
                delta_r_t = np.clip(prev_dr_ev + x_r, -DELTA_CLIP_R, DELTA_CLIP_R) * act_t
            else:
                delta_r_t = np.zeros(n_paths)

        dS_out[:, t] = delta_S_t.astype(np.float32)
        dr_out[:, t] = delta_r_t.astype(np.float32)

        R_t  = data['gross_ret'][:, t].astype(np.float64)
        mm_t = data['mm'][:, t].astype(np.float64)
        dsc  = data['disc'][:, t].astype(np.float64)

        gain = delta_S_t * A_t * (R_t - mm_t) * dsc
        if _eval_tc > 1e-12:
            gain -= _eval_tc * np.abs(delta_S_t - prev_dS_ev) * A_t * dsc * act_t

        if use_bond:
            ber  = data['bond_excess_ret'][:, t].astype(np.float64)
            gain += delta_r_t * ber * dsc
            if _eval_tc > 1e-12:
                bp_t_ev = data['bond_p'][:, t].astype(np.float64)
                gain -= _eval_tc * np.abs(delta_r_t - prev_dr_ev) * bp_t_ev * dsc * act_t

        pnl_running = (pnl_running + gain).astype(np.float32)

        prev_dS_ev = delta_S_t.astype(np.float32)
        prev_dr_ev = delta_r_t.astype(np.float32)

    pnl_final = pnl_running.astype(np.float64) - data['payout_disc'].astype(np.float64)

    return {
        'model':   model,
        'delta':   dS_out,
        'delta_r': dr_out,
        'delta_C': dC_out,
        'pnl':     pnl_final,
        **cvar_stats(pnl_final),
    }

In [ ]:
# -------------------------
# 12) Deep Hedge (Han-style) Method - HEDGING
# -------------------------

class HedgeSubnet(nn.Module):
    """Single time-step hedger with configurable input/output dimension."""
    def __init__(self, in_dim, out_dim, n_hidden=64, n_layers=2):
        super().__init__()
        lyrs = [nn.Linear(in_dim, n_hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            lyrs += [nn.Linear(n_hidden, n_hidden), nn.Tanh()]
        lyrs += [nn.Linear(n_hidden, out_dim)]
        self.net = nn.Sequential(*lyrs)
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)

class DeepHedger(nn.Module):
    """T independent subnetworks."""
    def __init__(self, T, in_dim, out_dim, n_hidden=64, n_layers=2):
        super().__init__()
        self.T    = T
        self.nets = nn.ModuleList(
            [HedgeSubnet(in_dim, out_dim, n_hidden, n_layers) for _ in range(T)])

    def forward(self, t, feat):
        return self.nets[t](feat)

def oce_cvar_loss(pnl, w, alpha=0.95):
    return w + torch.mean(torch.relu(-pnl - w)) / (1.0 - alpha)

def train_deep_hedge(data, p0, params, alpha=0.95, tc=0.0,
                     instruments='SB',
                     n_epochs=300, lr=1e-3, n_hidden=64, n_layers=2,
                     device='cpu', seed=0, verbose=True):
    """
    Train deep hedger minimising CVaR_alpha(-PnL).
    """
    torch.manual_seed(seed)
    T, P_nom = params.T_steps, float(params.P)
    v0 = max(float(getattr(params, 'theta_v', 0.0225)), 1e-6)
    r0 = max(float(params.theta_r), 1e-6)
    k0 = float(params.lc_model.k_t[-1])
    sk = (max(float(params.lc_model.volatility)*max(float(params.T)**0.5, 1.0), 1e-6)
          if params.use_stochastic_mortality else 1e-6)

    use_bond = 'B' in instruments

    DELTA_CLIP_S = 10.0
    DELTA_CLIP_R =  3.0 

    # Input/output dimensions
    in_dim  = 8 + (1 if use_bond else 0)
    out_dim = 1 + (1 if use_bond else 0)

    def _t(arr):
        return torch.tensor(arr, dtype=torch.float32, device=device)

    A_d   = _t(data['A'])
    W_d   = _t(data['W'])
    BD_d  = _t(data['BD'])
    v_d   = _t(data['v'])
    r_d   = _t(data['r'])
    k_d   = _t(data['k'])
    R_d   = _t(data['gross_ret'])
    mm_d = _t(data['mm'])
    dsc_d = _t(data['disc'])
    act_d = _t(data['active'])
    pay_d = _t(data['payout_disc'])
    bp_d  = _t(data['bond_p'])
    ber_d = _t(data['bond_excess_ret'])
    n = A_d.shape[0]

    model = DeepHedger(T, in_dim=in_dim, out_dim=out_dim,
                       n_hidden=n_hidden, n_layers=n_layers).to(device)
    w_var = torch.tensor(0.0, device=device, requires_grad=True)
    opt   = optim.Adam(list(model.parameters()) + [w_var], lr=lr)
    sched = optim.lr_scheduler.StepLR(opt, step_size=100, gamma=0.5)

    def _build_feat(t, prev_S, prev_r, prev_C):
        parts = [
            torch.full((n,), t/T, device=device),
            A_d[:, t]/P_nom, W_d[:, t]/P_nom, BD_d[:, t]/P_nom,
            torch.clamp(v_d[:, t]/v0, 0., 5.),
            torch.clamp(r_d[:, t]/r0, 0., 5.),
            torch.clamp((k_d[:, t] - k0)/sk, -3., 3.),
            prev_S,
        ]
        if use_bond:
            parts.append(prev_r)
        return torch.stack(parts, dim=1)

    for epoch in range(1, n_epochs + 1):
        model.train()
        pnl    = torch.full((n,), float(p0), device=device)
        prev_S = torch.zeros(n, device=device)
        prev_r = torch.zeros(n, device=device)
        prev_C = torch.zeros(n, device=device)

        for t in range(T):
            feat    = _build_feat(t, prev_S, prev_r, prev_C)
            out     = model(t, feat)
            act_t   = act_d[:, t]

            delta_S = torch.clamp(out[:, 0], -DELTA_CLIP_S, DELTA_CLIP_S) * act_t
            delta_r = torch.clamp(out[:, 1], -DELTA_CLIP_R, DELTA_CLIP_R) * act_t if use_bond else torch.zeros(n, device=device)

            gain  = delta_S * A_d[:, t] * (R_d[:, t] - mm_d[:, t]) * dsc_d[:, t]
            if use_bond:
                gain += delta_r * ber_d[:, t] * dsc_d[:, t]
            if tc > 0.0:
                gain -= tc * torch.abs(delta_S - prev_S) * A_d[:, t]  * dsc_d[:, t]
                if use_bond:
                    gain -= tc * torch.abs(delta_r - prev_r) * bp_d[:, t] * dsc_d[:, t]

            pnl    += gain
            prev_S  = delta_S.detach()
            prev_r  = delta_r.detach()

        pnl  -= pay_d
        loss  = oce_cvar_loss(pnl, w_var, alpha=alpha)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()

        if verbose and epoch % 50 == 0:
            with torch.no_grad():
                q_low = torch.quantile(-pnl, alpha).item()
                cvar  = (-pnl[(-pnl) >= q_low]).mean().item()
            print('    epoch {:3d}  loss={:8.1f}  CVaR={:8.1f}'.format(
                epoch, loss.item(), cvar))

    # Evaluation
    model.eval()
    with torch.no_grad():
        pnl    = torch.full((n,), float(p0), device=device)
        prev_S = torch.zeros(n, device=device)
        prev_r = torch.zeros(n, device=device)
        dS_out = torch.zeros(n, T, device=device)
        dr_out = torch.zeros(n, T, device=device)

        for t in range(T):
            feat    = _build_feat(t, prev_S, prev_r, prev_C)
            out     = model(t, feat)
            act_t   = act_d[:, t]

            delta_S = torch.clamp(out[:, 0], -DELTA_CLIP_S, DELTA_CLIP_S) * act_t
            delta_r = torch.clamp(out[:, 1], -DELTA_CLIP_R, DELTA_CLIP_R) * act_t if use_bond else torch.zeros(n, device=device)

            dS_out[:, t] = delta_S
            dr_out[:, t] = delta_r

            gain  = delta_S * A_d[:, t] * (R_d[:, t] - mm_d[:, t]) * dsc_d[:, t]
            if use_bond:
                gain += delta_r * ber_d[:, t] * dsc_d[:, t]
            if tc > 0.0:
                gain -= tc * torch.abs(delta_S - prev_S) * A_d[:, t]  * dsc_d[:, t]
                if use_bond:
                    gain -= tc * torch.abs(delta_r - prev_r) * bp_d[:, t] * dsc_d[:, t]
            pnl    += gain
            prev_S  = delta_S
            prev_r  = delta_r

        pnl   -= pay_d
        pnl_np = pnl.cpu().numpy()

    return {
        'model':   model,
        'delta':   dS_out.cpu().numpy(),
        'delta_r': dr_out.cpu().numpy(),
        'pnl':     pnl_np,
        **cvar_stats(pnl_np),
    }

In [ ]:
# -------------------------
# 13) Numerical Experiments - PRICING
# -------------------------
import time
import numpy as np
import pandas as pd

PINN = True

CASES = [
    {
        "name": "GMWB + GMDB",
        "flags": {"has_GMWB": True, "has_GMAB": False, "has_GMIB": False, "has_GMDB": True},
        "overrides": {
        },
    },
    {
        "name": "GMAB + GMDB",
        "flags": {"has_GMWB": False, "has_GMAB": True, "has_GMIB": False, "has_GMDB": True},
        "overrides": {
        },
    },
    {
        "name": "GMIB + GMDB",
        "flags": {"has_GMWB": False, "has_GMAB": False, "has_GMIB": True, "has_GMDB": True},
        "overrides": {  
        },
    },
]

PINN_HP_STOCH = {
    "GMWB+GMDB": dict(
        arch="valuenet",
        epochs=1000,
        pretrain_epochs=150,
        n_int=10000,
        n_T=10000,
        n_W0=10000,
        n_anni=5000,
        lr=5e-5,
        lr_step=250,
        lr_gamma=0.82,
        w_pde=40.0,
        w_T=80.0,
        w_W0=50.0,
        w_anni=50.0,
    ),
    "GMAB+GMDB": dict(
        arch="valuenet",   
        epochs=1000,        
        pretrain_epochs=150,
        n_int=20000,
        n_T=26000,
        n_W0=10000,
        n_anni=12000,
        lr=5e-5,
        lr_step=250,
        lr_gamma=0.82,
        w_pde=40.0,
        w_T=80.0,
        w_W0=50.0,
        w_anni=50.0,
    ),
    "GMIB+GMDB": dict(
        arch="valuenet",
        epochs=1000,
        pretrain_epochs=150,
        n_int=20000,
        n_T=26000,
        n_W0=10000,
        n_anni=12000,
        lr=5e-5,
        lr_step=250,
        lr_gamma=0.82,
        w_pde=60.0,
        w_T=80.0,
        w_W0=50.0,
        w_anni=0.5,     
    ),
}


def _contract_bucket_from_flags(flags):
    if flags.get("has_GMWB", False):
        return "GMWB+GMDB"
    if flags.get("has_GMIB", False):
        return "GMIB+GMDB"
    return "GMAB+GMDB"


def build_params(case):
    p = VAParams(
        dt=1.0,
        lc_model=lc_model,
        use_stochastic_rates=True,
        use_stochastic_mortality=True,
    )
    p.has_GMWB = case["flags"]["has_GMWB"]
    p.has_GMAB = case["flags"]["has_GMAB"]
    p.has_GMIB = case["flags"]["has_GMIB"]
    p.has_GMDB = case["flags"]["has_GMDB"]

    for k, v in case["overrides"].items():
        setattr(p, k, v)

    p.gmw_step_frac = p.gmw_annual_frac * p.dt
    p.jump_compensator = p.lam * (np.exp(p.jump_mean + 0.5 * p.jump_std**2) - 1)
    return p

def _extract_dp_t0_value(dpres, params_local):
    k0 = float(params_local.lc_model.k_t[-1])

    if params_local.has_GMWB:
        keys = ["A_grid", "G_grid"]
        vals = [params_local.P, params_local.gmw_total]
        if ("B_grid" in dpres) and params_local.has_GMDB:
            keys.append("B_grid")
            vals.append(params_local.P)
        keys += ["V_grid", "R_grid", "K_grid"]
        vals += [params_local.v0, params_local.r0, k0]
    else:
        keys = ["A_grid", "V_grid", "R_grid", "K_grid"]
        vals = [params_local.P, params_local.v0, params_local.r0, k0]

    idx = [int(np.argmin(np.abs(np.asarray(dpres[key]) - val))) for key, val in zip(keys, vals)]
    return float(dpres["V"][(0, *idx)])


PRICING_ROWS = []
HAN_MODELS = {}
PARAMS = {}

for case in CASES:
    cname = case["name"]
    print("\n" + "=" * 88)
    print(f"PRICING CASE: {cname}")
    print("=" * 88)

    p_case = build_params(case)
    PARAMS[cname] = p_case

    # DP method
    if p_case.has_GMWB:
        dp_cfg = dict(n_A=5, n_G=5, n_B=5, n_V=5, n_R=5, n_K=5, n_return_samples=1000)
    else:
        dp_cfg = dict(n_A=20, n_V=20, n_R=20, n_K=20, n_return_samples=1000)

    t0 = time.time()
    dp_case = dp_solver(p_case, **dp_cfg)
    dp_val = _extract_dp_t0_value(dp_case, p_case)
    dp_time = time.time() - t0

    PRICING_ROWS.append({
        "contract": cname,
        "method": "DP (coarse)",
        "price": dp_val,
        "time_s": dp_time,
    })
    print(f"  DP (coarse):      price={dp_val:,.2f} | time={dp_time:.1f}s")

    # Han method
    t0 = time.time()
    han_case = HanPolicy(p_case, hidden_units=256, learning_rate=1e-3, hidden_layers=3)
    han_case.train(p_case, epochs=50, n_paths=15000, verbose=False)
    han_val = evaluate_han_policy_mc(han_case, p_case, n_paths=150000, seed=2026)
    han_time = time.time() - t0

    HAN_MODELS[cname] = han_case
    PRICING_ROWS.append({
        "contract": cname,
        "method": "Han (MC eval)",
        "price": float(han_val),
        "time_s": han_time,
    })
    print(f"  Han (MC eval):    price={han_val:,.2f} | time={han_time:.1f}s")

    # Pinn method
    if PINN:
        bucket = _contract_bucket_from_flags(case["flags"])
        hp = PINN_HP_STOCH[bucket]

        t0 = time.time()
        pinn_model_case, pinn_val_case = train_pinn(
            p_case,
            arch=hp["arch"],
            epochs=hp["epochs"],
            pretrain_epochs=hp["pretrain_epochs"],
            n_int=hp["n_int"],
            n_T=hp["n_T"],
            n_W0=hp["n_W0"],
            n_anni=hp["n_anni"],
            n_annuity_paths=hp["n_anni"],
            use_stochastic_mortality=True,
            lr=hp["lr"],
            lr_step=hp["lr_step"],
            lr_gamma=hp["lr_gamma"],
            w_T=hp["w_T"],
            w_W0=hp["w_W0"],
            w_anni=hp["w_anni"],
            w_pde=hp["w_pde"],
            n_gamma=10,
            alpha_v=p_case.sigma_v,
        )
        pinn_time = time.time() - t0

        PRICING_ROWS.append({
            "contract": cname,
            "method": "PINN (thesis hp)",
            "price": float(pinn_val_case),
            "time_s": pinn_time,
        })
        print(f"  PINN ({bucket}):  price={pinn_val_case:,.2f} | time={pinn_time:.1f}s")

PRICING_TABLE = pd.DataFrame(PRICING_ROWS)
print("\n" + "=" * 88)
print("PRICING RESULTS")
print("=" * 88)
print(PRICING_TABLE.pivot_table(index="contract", columns="method", values="price", aggfunc="first").round(2))
print("\nRuntime summary (seconds):")
print(PRICING_TABLE.pivot_table(index="contract", columns="method", values="time_s", aggfunc="first").round(1))

In [ ]:
# -------------------------
# 14) Numerical Experiments - HEDGING
# -------------------------
import time
import numpy as np
import pandas as pd

HEDGE_N_PATHS = 6000
HEDGE_ALPHA = 0.95
HEDGE_TC = [0.0, 0.005]
HEDGE_PHASES = ["A", "B"]
HEDGE_INSTR = {"A": "S", "B": "SB"}
RUN_DP = True

HEDGE_CASE_NAMES = [
    "GMAB + GMDB",
    "GMIB + GMDB",
    "GMWB + GMDB",
]

if "PRICING_TABLE" not in globals():
    raise RuntimeError("Please run pricing experiments before running this hedging cell.")


def _get_reference_price(contract_name):
    sub = PRICING_TABLE[PRICING_TABLE["contract"] == contract_name]
    if len(sub) == 0:
        return None
    dp_sub = sub[sub["method"] == "DP (coarse)"]
    if len(dp_sub) > 0:
        return float(dp_sub["price"].iloc[0])
    return float(sub["price"].mean())


def _extra_dp_cfg(params_local):
    if params_local.has_GMWB:
        return dict(n_A=5, n_G=5, n_B=5, n_V=5, n_R=5, n_K=5, n_return_samples=1000)
    return dict(n_A=20, n_V=20, n_R=20, n_K=20, n_return_samples=1000)


HEDGE_ROWS = []

for cname in HEDGE_CASE_NAMES:
    if cname not in PARAMS:
        print(f"Skipping {cname}: params not found from pricing cell.")
        continue

    params_h = PARAMS[cname]
    p0_h = _get_reference_price(cname)

    if p0_h is None:
        print(f"Skipping {cname}: no pricing reference.")
        continue

    print("\n" + "=" * 92)
    print(f"HEDGING CASE: {cname}")
    print("=" * 92)
    print(f"Using liability premium p0 = {p0_h:,.2f}")

    han_h = HAN_MODELS.get(cname, None)
    if han_h is None and (params_h.has_GMWB or params_h.has_GMIB):
        print("  Han model not found from pricing cell; training a quick policy for behavior simulation...")
        han_h = HanPolicy(params_h, hidden_units=256, learning_rate=1e-3, hidden_layers=3)
        han_h.train(params_h, epochs=50, n_paths=15000, verbose=False)
        HAN_MODELS[cname] = han_h

    t0 = time.time()
    data_h = simulate_hedge_paths(params_h, han_h, HEDGE_N_PATHS, seed=2026)
    sim_time = time.time() - t0
    print(f"  Simulated {HEDGE_N_PATHS:,} paths in {sim_time:.1f}s")

    t0 = time.time()
    pnl_unh = compute_pnl(np.zeros_like(data_h["A"]), data_h, p0_h, params_h, tc=0.0)
    base_stats = cvar_stats(pnl_unh, alpha=HEDGE_ALPHA)
    base_time = time.time() - t0

    HEDGE_ROWS.append({
        "contract": cname,
        "phase": "--",
        "method": "Unhedged",
        "tc": 0.0,
        "mean_pnl": base_stats["mean_pnl"],
        "std_pnl": base_stats["std_pnl"],
        "cvar95": base_stats["cvar95"],
        "time_s": base_time,
    })

    if RUN_DP:
        print("  Running DP delta hedge...")
        t0 = time.time()
        dp_h = dp_solver(params_h, **_extra_dp_cfg(params_h))
        dp_solve_time = time.time() - t0

        for tc in HEDGE_TC:
            t1 = time.time()
            dp_full = dp_delta_hedge(dp_h, data_h, p0_h, params_h, tc=tc)
            dp_eval_time = time.time() - t1

            # Phase A
            pnl_A = compute_pnl(dp_full["delta"], data_h, p0_h, params_h, tc=tc)
            stats_A = cvar_stats(pnl_A, alpha=HEDGE_ALPHA)
            HEDGE_ROWS.append({
                "contract": cname,
                "phase": "A",
                "method": "DP delta",
                "tc": float(tc),
                "mean_pnl": stats_A["mean_pnl"],
                "std_pnl": stats_A["std_pnl"],
                "cvar95": stats_A["cvar95"],
                "time_s": dp_solve_time + dp_eval_time,
            })

            # Phase B
            HEDGE_ROWS.append({
                "contract": cname,
                "phase": "B",
                "method": "DP delta",
                "tc": float(tc),
                "mean_pnl": float(dp_full["mean_pnl"]),
                "std_pnl": float(dp_full["std_pnl"]),
                "cvar95": float(dp_full["cvar95"]),
                "time_s": dp_solve_time + dp_eval_time,
            })

    for phase in HEDGE_PHASES:
        instr = HEDGE_INSTR[phase]

        t0 = time.time()
        hjb_0 = train_hjb_pinn_hedge(
            data_h,
            p0_h,
            params_h,
            alpha=HEDGE_ALPHA,
            instruments=instr,
            tc=0.0,
            device="cpu",
            seed=2026,
            verbose=False,
            n_epochs=1000,
            pretrain_epochs=150,
            lr=5e-5,
            lr_step=200,
            lr_gamma=0.85,
            n_pde=5000,
            n_term=5000,
            n_hidden=128,
            n_layers=3,
            w_pde=40.0,
            w_term=80.0,
        )
        hjb_train_time = time.time() - t0

        HEDGE_ROWS.append({
            "contract": cname,
            "phase": phase,
            "method": "HJB PINN",
            "tc": 0.0,
            "mean_pnl": float(hjb_0["mean_pnl"]),
            "std_pnl": float(hjb_0["std_pnl"]),
            "cvar95": float(hjb_0["cvar95"]),
            "time_s": hjb_train_time,
        })

        for tc in [x for x in HEDGE_TC if x > 0.0]:
            t0 = time.time()
            hjb_tc = train_hjb_pinn_hedge(
                data_h,
                p0_h,
                params_h,
                alpha=HEDGE_ALPHA,
                instruments=instr,
                tc=0.0,
                eval_tc=tc,
                device="cpu",
                seed=2026,
                verbose=False,
                n_epochs=0,
                pretrain_epochs=0,
                init_model=hjb_0["model"],
            )
            ev_time = time.time() - t0
            HEDGE_ROWS.append({
                "contract": cname,
                "phase": phase,
                "method": "HJB PINN",
                "tc": float(tc),
                "mean_pnl": float(hjb_tc["mean_pnl"]),
                "std_pnl": float(hjb_tc["std_pnl"]),
                "cvar95": float(hjb_tc["cvar95"]),
                "time_s": ev_time,
            })

        for tc in HEDGE_TC:
            t0 = time.time()
            deep_res = train_deep_hedge(
                data_h,
                p0_h,
                params_h,
                alpha=HEDGE_ALPHA,
                tc=tc,
                instruments=instr,
                n_epochs=50,
                lr=1e-3,
                n_hidden=256,
                n_layers=3,
                device="cpu",
                seed=2026,
                verbose=False,
            )
            deep_time = time.time() - t0
            HEDGE_ROWS.append({
                "contract": cname,
                "phase": phase,
                "method": "Deep hedge",
                "tc": float(tc),
                "mean_pnl": float(deep_res["mean_pnl"]),
                "std_pnl": float(deep_res["std_pnl"]),
                "cvar95": float(deep_res["cvar95"]),
                "time_s": deep_time,
            })

HEDGE_TABLE = pd.DataFrame(HEDGE_ROWS)

print("\n" + "=" * 92)
print("HEDGING RESULTS")
print("=" * 92)

show_cols = ["contract", "phase", "method", "tc", "mean_pnl", "std_pnl", "cvar95", "time_s"]
print(HEDGE_TABLE[show_cols].sort_values(["contract", "phase", "method", "tc"]).round(2))

print("\nCompact CVaR table:")
print(
    HEDGE_TABLE
    .pivot_table(index=["contract", "phase"], columns=["method", "tc"], values="cvar95", aggfunc="first")
    .round(2)
)

In [ ]:
# -------------------------
# 15) Numerical Experiments (Riders-only) - PRICING AND HEDGING
# -------------------------
import time
import numpy as np
import pandas as pd

RUN_RIDER_PRICING = True
RUN_RIDER_PINN = True
RUN_RIDER_HEDGING = True

RIDER_ONLY_CASES = [
    {
        "name": "GMDB only",
        "flags": {"has_GMWB": False, "has_GMAB": False, "has_GMIB": False, "has_GMDB": True},
        "overrides": {
        },
    },
    {
        "name": "GMAB only",
        "flags": {"has_GMWB": False, "has_GMAB": True,  "has_GMIB": False, "has_GMDB": False},
        "overrides": {
        },
    },
    {
        "name": "GMIB only",
        "flags": {"has_GMWB": False, "has_GMAB": False, "has_GMIB": True,  "has_GMDB": False},
        "overrides": {
        },
    },
    {
        "name": "GMWB only",
        "flags": {"has_GMWB": True,  "has_GMAB": False, "has_GMIB": False, "has_GMDB": False},
        "overrides": {
        },
    },
]

if "PINN_HP_STOCH" not in globals():
    PINN_HP_STOCH = {
        "GMAB+GMDB": dict(arch="valuenet", epochs=1000, pretrain_epochs=150, n_int=20000, n_T=26000, n_W0=10000, n_anni=12000, lr=5e-5, lr_step=250, lr_gamma=0.82, w_pde=40.0, w_T=80.0, w_W0=50.0, w_anni=50.0),
        "GMIB+GMDB": dict(arch="valuenet", epochs=1000, pretrain_epochs=150, n_int=20000, n_T=26000, n_W0=10000, n_anni=12000, lr=5e-5, lr_step=250, lr_gamma=0.82, w_pde=60.0, w_T=80.0, w_W0=50.0, w_anni=0.5),
        "GMWB+GMDB": dict(arch="valuenet", epochs=1000, pretrain_epochs=150, n_int=10000, n_T=10000, n_W0=10000, n_anni=5000, lr=5e-5, lr_step=250, lr_gamma=0.82, w_pde=40.0, w_T=80.0, w_W0=50.0, w_anni=50.0),
    }


def _rider_bucket(flags):
    if flags.get("has_GMWB", False):
        return "GMWB+GMDB"
    if flags.get("has_GMIB", False):
        return "GMIB+GMDB"
    return "GMAB+GMDB"


def _build_rider_params(case):
    p = VAParams(
        dt=1.0,
        lc_model=lc_model,
        use_stochastic_rates=True,
        use_stochastic_mortality=True,
    )
    p.has_GMWB = case["flags"]["has_GMWB"]
    p.has_GMAB = case["flags"]["has_GMAB"]
    p.has_GMIB = case["flags"]["has_GMIB"]
    p.has_GMDB = case["flags"]["has_GMDB"]

    for k, v in case["overrides"].items():
        setattr(p, k, v)

    p.gmw_step_frac = p.gmw_annual_frac * p.dt
    p.jump_compensator = p.lam * (np.exp(p.jump_mean + 0.5 * p.jump_std**2) - 1)
    return p

def _extract_dp_price(dpres, params_local):
    k0 = float(params_local.lc_model.k_t[-1])
    if params_local.has_GMWB:
        keys = ["A_grid", "G_grid", "V_grid", "R_grid", "K_grid"]
        vals = [params_local.P, params_local.gmw_total, params_local.v0, params_local.r0, k0]
        if ("B_grid" in dpres) and params_local.has_GMDB:
            keys = ["A_grid", "G_grid", "B_grid", "V_grid", "R_grid", "K_grid"]
            vals = [params_local.P, params_local.gmw_total, params_local.P, params_local.v0, params_local.r0, k0]
    else:
        keys = ["A_grid", "V_grid", "R_grid", "K_grid"]
        vals = [params_local.P, params_local.v0, params_local.r0, k0]
    idx = [int(np.argmin(np.abs(np.asarray(dpres[k]) - v))) for k, v in zip(keys, vals)]
    return float(dpres["V"][(0, *idx)])


def _dp_cfg_for_case(params_local):
    if params_local.has_GMWB:
        return dict(n_A=5, n_G=5, n_B=5, n_V=5, n_R=5, n_K=5, n_return_samples=1000)
    return dict(n_A=20, n_V=20, n_R=20, n_K=20, n_return_samples=1000)

RIDER_PARAMS = globals().get("RIDER_PARAMS", {})
RIDER_HAN_MODELS = globals().get("RIDER_HAN_MODELS", {})

if RUN_RIDER_PRICING:
    RIDER_PRICING_ROWS = []

    for case in RIDER_ONLY_CASES:
        cname = case["name"]
        p_case = _build_rider_params(case)
        RIDER_PARAMS[cname] = p_case

        print("\n" + "=" * 88)
        print(f"RIDER-ONLY PRICING CASE: {cname}")
        print("=" * 88)

        t0 = time.time()
        dp_case = dp_solver(p_case, **_dp_cfg_for_case(p_case))
        dp_price = _extract_dp_price(dp_case, p_case)
        dp_time = time.time() - t0
        RIDER_PRICING_ROWS.append({"contract": cname, "method": "DP", "price": dp_price, "time_s": dp_time})
        print(f"  DP:          price={dp_price:,.2f} | time={dp_time:.1f}s")

        t0 = time.time()
        han_case = HanPolicy(p_case, hidden_units=256, learning_rate=1e-3, hidden_layers=3)
        han_case.train(p_case, epochs=50, n_paths=15000, verbose=False)
        han_price = evaluate_han_policy_mc(han_case, p_case, n_paths=150000, seed=2027)
        han_time = time.time() - t0
        RIDER_HAN_MODELS[cname] = han_case
        RIDER_PRICING_ROWS.append({"contract": cname, "method": "Han", "price": float(han_price), "time_s": han_time})
        print(f"  Han:         price={han_price:,.2f} | time={han_time:.1f}s")

        if RUN_RIDER_PINN:
            hp = PINN_HP_STOCH[_rider_bucket(case["flags"])]
            t0 = time.time()
            _, pinn_price = train_pinn(
                p_case,
                arch=hp["arch"],
                epochs=hp["epochs"],
                pretrain_epochs=hp["pretrain_epochs"],
                n_int=hp["n_int"],
                n_T=hp["n_T"],
                n_W0=hp["n_W0"],
                n_anni=hp["n_anni"],
                n_annuity_paths=hp["n_anni"],
                use_stochastic_mortality=True,
                lr=hp["lr"],
                lr_step=hp["lr_step"],
                lr_gamma=hp["lr_gamma"],
                w_T=hp["w_T"],
                w_W0=hp["w_W0"],
                w_anni=hp["w_anni"],
                w_pde=hp["w_pde"],
                n_gamma=150,
                alpha_v=p_case.sigma_v,
            )
            pinn_time = time.time() - t0
            RIDER_PRICING_ROWS.append({"contract": cname, "method": "PINN", "price": float(pinn_price), "time_s": pinn_time})
            print(f"  PINN:        price={pinn_price:,.2f} | time={pinn_time:.1f}s")

    RIDER_PRICING_TABLE = pd.DataFrame(RIDER_PRICING_ROWS)
    print("\n" + "=" * 88)
    print("RIDER-ONLY PRICING RESULTS")
    print("=" * 88)
    print(RIDER_PRICING_TABLE.pivot_table(index="contract", columns="method", values="price", aggfunc="first").round(2))
    print("\nRIDER-ONLY PRICING RUNTIME (s)")
    print(RIDER_PRICING_TABLE.pivot_table(index="contract", columns="method", values="time_s", aggfunc="first").round(1))
else:
    print("Skipping rider-only pricing (RUN_RIDER_PRICING=False).")

    for case in RIDER_ONLY_CASES:
        if case["name"] not in RIDER_PARAMS:
            RIDER_PARAMS[case["name"]] = _build_rider_params(case)

    if "RIDER_PRICING_TABLE" not in globals():
        # FALLBACK PRICES (only if pricing is not run)
        _fallback_dp_p0 = {
            "GMAB only": 8438.67,
            "GMIB only": 15193.85,
            "GMWB only": 12073.45,
            "GMDB only": 11438.33,
        }
        RIDER_PRICING_TABLE = pd.DataFrame(
            [{"contract": k, "method": "DP", "price": v, "time_s": np.nan} for k, v in _fallback_dp_p0.items()]
        )
        print("RIDER_PRICING_TABLE not found in memory; using fallback DP prices from prior run.")

if RUN_RIDER_HEDGING:
    RIDER_HEDGE_ROWS = []
    HEDGE_TC = [0.0, 0.005]
    HEDGE_PHASES = ["A", "B"]
    INSTR = {"A": "S", "B": "SB"}

    for case in RIDER_ONLY_CASES:
        cname = case["name"]
        p_case = RIDER_PARAMS[cname]

        p0_sub = RIDER_PRICING_TABLE[(RIDER_PRICING_TABLE["contract"] == cname) & (RIDER_PRICING_TABLE["method"] == "DP")]
        if len(p0_sub) == 0:
            raise RuntimeError(f"Missing DP price in RIDER_PRICING_TABLE for case: {cname}")
        p0 = float(p0_sub["price"].iloc[0])

        han_model = RIDER_HAN_MODELS.get(cname, None)

        print("\n" + "-" * 88)
        print(f"RIDER-ONLY HEDGING CASE: {cname} | p0={p0:,.2f}")
        print("-" * 88)

        t0 = time.time()
        data_h = simulate_hedge_paths(p_case, han_model, n_paths=5000, seed=2027)
        print(f"  path simulation time: {time.time() - t0:.1f}s")

        pnl_unh = compute_pnl(np.zeros_like(data_h["A"]), data_h, p0, p_case, tc=0.0)
        stats_unh = cvar_stats(pnl_unh)
        RIDER_HEDGE_ROWS.append({
            "contract": cname, "phase": "--", "method": "Unhedged", "tc": 0.0,
            "mean_pnl": stats_unh["mean_pnl"], "std_pnl": stats_unh["std_pnl"], "cvar95": stats_unh["cvar95"]
        })

        t0 = time.time()
        dp_h = dp_solver(p_case, **_dp_cfg_for_case(p_case))
        dp_solve_time = time.time() - t0

        for tc in HEDGE_TC:
            dp_res = dp_delta_hedge(dp_h, data_h, p0, p_case, tc=tc)
            pnl_A = compute_pnl(dp_res["delta"], data_h, p0, p_case, tc=tc)
            stats_A = cvar_stats(pnl_A)

            RIDER_HEDGE_ROWS.append({
                "contract": cname, "phase": "A", "method": "DP delta", "tc": tc,
                "mean_pnl": stats_A["mean_pnl"], "std_pnl": stats_A["std_pnl"], "cvar95": stats_A["cvar95"],
                "time_s": dp_solve_time
            })
            RIDER_HEDGE_ROWS.append({
                "contract": cname, "phase": "B", "method": "DP delta", "tc": tc,
                "mean_pnl": float(dp_res["mean_pnl"]), "std_pnl": float(dp_res["std_pnl"]), "cvar95": float(dp_res["cvar95"]),
                "time_s": dp_solve_time
            })

        for phase in HEDGE_PHASES:
            instr = INSTR[phase]

            hjb0 = train_hjb_pinn_hedge(
                data_h, p0, p_case, alpha=0.95, instruments=instr, tc=0.0,
                device="cpu", seed=2027, verbose=False,
                n_epochs=1000, pretrain_epochs=150,
                lr=5e-5, lr_step=200, lr_gamma=0.85,
                n_pde=5000, n_term=5000, n_hidden=128, n_layers=3,
                w_pde=40.0, w_term=80.0,
            )
            RIDER_HEDGE_ROWS.append({
                "contract": cname, "phase": phase, "method": "HJB PINN", "tc": 0.0,
                "mean_pnl": float(hjb0["mean_pnl"]), "std_pnl": float(hjb0["std_pnl"]), "cvar95": float(hjb0["cvar95"])
            })

            hjb_tc = train_hjb_pinn_hedge(
                data_h, p0, p_case, alpha=0.95, instruments=instr, tc=0.0, eval_tc=0.005,
                device="cpu", seed=2027, verbose=False,
                n_epochs=0, pretrain_epochs=0, init_model=hjb0["model"],
            )
            RIDER_HEDGE_ROWS.append({
                "contract": cname, "phase": phase, "method": "HJB PINN", "tc": 0.005,
                "mean_pnl": float(hjb_tc["mean_pnl"]), "std_pnl": float(hjb_tc["std_pnl"]), "cvar95": float(hjb_tc["cvar95"])
            })

            for tc in HEDGE_TC:
                deep_res = train_deep_hedge(
                    data_h, p0, p_case, alpha=0.95, tc=tc, instruments=instr,
                    n_epochs=50, lr=1e-3, n_hidden=256, n_layers=3,
                    device="cpu", seed=2027, verbose=False,
                )
                RIDER_HEDGE_ROWS.append({
                    "contract": cname, "phase": phase, "method": "Deep hedge", "tc": tc,
                    "mean_pnl": float(deep_res["mean_pnl"]), "std_pnl": float(deep_res["std_pnl"]), "cvar95": float(deep_res["cvar95"])
                })

    RIDER_HEDGE_TABLE = pd.DataFrame(RIDER_HEDGE_ROWS)
    print("\n" + "=" * 88)
    print("RIDER-ONLY HEDGING RESULTS (CVaR95)")
    print("=" * 88)
    print(RIDER_HEDGE_TABLE.sort_values(["contract", "phase", "method", "tc"]).round(2))
    print("\nCompact CVaR table:")
    print(
        RIDER_HEDGE_TABLE
        .pivot_table(index=["contract", "phase"], columns=["method", "tc"], values="cvar95", aggfunc="first")
        .round(2)
    )